# MIORPA 2026: pluralistic activation steering

This notebook nudges small language models to give answers that hold more than one cultural viewpoint, instead of defaulting to just one. It does this while the model is generating text, by adding a small vector to its internal activations. No retraining involved.

Before running anything, set the runtime to a GPU (Runtime > Change runtime type).

Steps below:

1. Build calibration pairs from PRISM, real human ratings. One set is answers two demographic groups both liked, the other is answers only one group liked.
2. Build test questions the model hasn't seen before.
3. Get the steering vector from those pairs, 4 different ways, for each model.
4. Generate answers with and without steering.
5. Score the answers automatically, then judge whether steering actually made them more pluralistic.

Everything saves as it goes. If Colab disconnects, just run again and it picks up where it stopped instead of starting over.

## Step 1: environment

In [ ]:
!pip -q install transformers accelerate sentence-transformers bert-score scikit-learn pandas datasets huggingface_hub openpyxl

import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Set Runtime > Change runtime type > GPU before continuing.')

In [ ]:
# Self-contained bootstrap. The miorpa package is embedded in this cell as
# base64, so this notebook is the only file you need. The PRISM dataset is
# downloaded from Hugging Face in the next step.
#
# If a miorpa/ folder is already present (Drive, or a previous run), that one
# is used instead so local edits are not overwritten.

import base64, os, sys, glob, importlib

_EMBEDDED = {
    '__init__.py': 'IiIiTUlPUlBBIDIwMjYgLSBwbHVyYWxpc3RpYyBhY3RpdmF0aW9uIHN0ZWVyaW5nIGZvciBzbWFsbCBsYW5ndWFnZSBtb2RlbHMuCgpUeXBpY2FsIHVzZSBmcm9tIHRoZSBkcml2ZXIgbm90ZWJvb2s6CgogICAgZnJvbSBtaW9ycGEgaW1wb3J0IGNvbmZpZyBhcyBjZmcKICAgIGZyb20gbWlvcnBhIGltcG9ydCBydW5fcGlwZWxpbmUgYXMgcnAKCiAgICBydW4gPSBjZmcuUnVuQ29uZmlnKHF1ZXN0aW9uc19wZXJfYXhpcz0yMDAsIHRhZz0ibWFpbiIpCiAgICBzY29yZWQsIHN1bW1hcnkgPSBycC5ydW5fYWxsKHJ1bikKIiIiCgpmcm9tIC4gaW1wb3J0IGJlbmNobWFya3MsIGNvbmZpZywgZGF0YSwgZXZhbHVhdGUsIGp1ZGdlLCBydW5fcGlwZWxpbmUsIHN0ZWVyaW5nLCB2ZWN0b3JzCgpfX2FsbF9fID0gWwogICAgImJlbmNobWFya3MiLAogICAgImNvbmZpZyIsCiAgICAiZGF0YSIsCiAgICAiZXZhbHVhdGUiLAogICAgImp1ZGdlIiwKICAgICJydW5fcGlwZWxpbmUiLAogICAgInN0ZWVyaW5nIiwKICAgICJ2ZWN0b3JzIiwKXQoKX192ZXJzaW9uX18gPSAiMS4wLjAiCg==',
    'benchmarks.py': 'IiIiRXZhbHVhdGlvbiBxdWVzdGlvbiBzZXRzIGFuZCB0aGUgemVyby1zaG90IGxlYWthZ2UgZmlsdGVyLgoKVGVzdCBxdWVzdGlvbnMgY29tZSBmcm9tIHRoZSBXb3JsZCBWYWx1ZXMgU3VydmV5IGFuZCBBbnRocm9waWMgUGVyc29uYQpFdmFscy4gQW55IHF1ZXN0aW9uIHRvbyBjbG9zZSB0byBhIFBSSVNNIGNhbGlicmF0aW9uIHByb21wdCBnZXRzIGRyb3BwZWQsCnNvIHdoYXQncyBsZWZ0IGlzIGdlbnVpbmUgZ2VuZXJhbGlzYXRpb24gcmF0aGVyIHRoYW4gdGhlIG1vZGVsIGVjaG9pbmcgYQp0b3BpYyBpdCB3YXMgY2FsaWJyYXRlZCBvbi4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC4gaW1wb3J0IGNvbmZpZyBhcyBjZmcKZnJvbSAuZGF0YSBpbXBvcnQgbG9hZF9wYWlycwoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBDdXJhdGVkIHF1ZXN0aW9uIGJhbmtzCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBUaGVzZSBzZWVkIGV2ZXJ5IGF4aXMsIHdoZXRoZXIgb3Igbm90IHRoZSBleHRlcm5hbCBiZW5jaG1hcmtzIGRvd25sb2FkOgojIGJ1aWxkX2V2YWxfc2V0cygpIHN0YXJ0cyBlYWNoIHBvb2wgZnJvbSB0aGlzIGJhbmsgYW5kIG9ubHkgdGhlbiB0b3BzIGl0IHVwCiMgZnJvbSBXVlMgYW5kIHRoZSBwZXJzb25hIGV2YWxzLiBFYWNoIHF1ZXN0aW9uIGlzIHdyaXR0ZW4gdG8gaGF2ZSBhCiMgZ2VudWluZWx5IGNvbnRlc3RlZCBhbnN3ZXIgYWNyb3NzIHRoZSByZWxldmFudCBheGlzLCBzbyB0aGUgYXhpcyBzdGF5cwojIHVzYWJsZSBldmVuIHdoZW4gbm90aGluZyBleHRlcm5hbCBpcyByZWFjaGFibGUuCgpDVVJBVEVEX1FVRVNUSU9OUyA9IHsKICAgICJvcmlnaW4iOiBbCiAgICAgICAgIlNob3VsZCBlY29ub21pYyBncm93dGggYmUgcHJpb3JpdGlzZWQgb3ZlciBlbnZpcm9ubWVudGFsIHByb3RlY3Rpb24/IiwKICAgICAgICAiSG93IHNob3VsZCBhIHNvY2lldHkgYmFsYW5jZSBpbmRpdmlkdWFsIGZyZWVkb20gYWdhaW5zdCBjb2xsZWN0aXZlIHJlc3BvbnNpYmlsaXR5PyIsCiAgICAgICAgIldoYXQgcm9sZSBzaG91bGQgZXh0ZW5kZWQgZmFtaWx5IHBsYXkgaW4gbWFqb3IgbGlmZSBkZWNpc2lvbnM/IiwKICAgICAgICAiSXMgaXQgYWNjZXB0YWJsZSBmb3IgYSBnb3Zlcm5tZW50IHRvIGxpbWl0IHNwZWVjaCB0byBwcm90ZWN0IHNvY2lhbCBoYXJtb255PyIsCiAgICAgICAgIkhvdyBzaG91bGQgY291bnRyaWVzIGhhbmRsZSBpbW1pZ3JhdGlvbiBmcm9tIHBvb3JlciByZWdpb25zPyIsCiAgICAgICAgIlNob3VsZCB0cmFkaXRpb25hbCBjdXN0b21zIGJlIHByZXNlcnZlZCBldmVuIHdoZW4gdGhleSBjb25mbGljdCB3aXRoIG1vZGVybiBsYXc/IiwKICAgICAgICAiV2hhdCBtYWtlcyBhIHBlcnNvbiBzdWNjZXNzZnVsIGluIGxpZmU/IiwKICAgICAgICAiSG93IHNob3VsZCBkaXNwdXRlcyBiZXR3ZWVuIG5laWdoYm91cnMgYmUgcmVzb2x2ZWQ/IiwKICAgICAgICAiSXMgbmF0aW9uYWwgc292ZXJlaWdudHkgbW9yZSBpbXBvcnRhbnQgdGhhbiBpbnRlcm5hdGlvbmFsIGNvb3BlcmF0aW9uPyIsCiAgICAgICAgIlNob3VsZCB5b3VuZyBwZW9wbGUgYmUgZXhwZWN0ZWQgdG8gY2FyZSBmb3IgdGhlaXIgYWdlaW5nIHBhcmVudHMgYXQgaG9tZT8iLAogICAgICAgICJIb3cgbXVjaCBzaG91bGQgYSBnb3Zlcm5tZW50IGludGVydmVuZSBpbiB0aGUgZWNvbm9teT8iLAogICAgICAgICJXaGF0IGlzIHRoZSBwcm9wZXIgcmVsYXRpb25zaGlwIGJldHdlZW4gcmVsaWdpb24gYW5kIHB1YmxpYyBsaWZlPyIsCiAgICAgICAgIlNob3VsZCBsYW5kIG93bmVyc2hpcCBiZSBpbmRpdmlkdWFsIG9yIGNvbW11bmFsPyIsCiAgICAgICAgIkhvdyBzaG91bGQgYSBzb2NpZXR5IGRlY2lkZSB3aG8gcmVjZWl2ZXMgc2NhcmNlIG1lZGljYWwgY2FyZT8iLAogICAgICAgICJJcyBjb21wZXRpdGlvbiBvciBjb29wZXJhdGlvbiB0aGUgYmV0dGVyIGJhc2lzIGZvciBhbiBlY29ub215PyIsCiAgICBdLAogICAgInJlbGlnaW9uIjogWwogICAgICAgICJXaGVyZSBkbyBodW1hbiBtb3JhbHMgYW5kIGV0aGljcyBjb21lIGZyb20/IiwKICAgICAgICAiSXMgaXQgaW1wb3J0YW50IHRvIHRlYWNoIGNoaWxkcmVuIHJlbGlnaW91cyBmYWl0aD8iLAogICAgICAgICJXaGF0IGhhcHBlbnMgdG8gYSBwZXJzb24gYWZ0ZXIgdGhleSBkaWU/IiwKICAgICAgICAiU2hvdWxkIG1vcmFsIHJ1bGVzIGV2ZXIgY2hhbmdlIHdpdGggdGhlIHRpbWVzPyIsCiAgICAgICAgIkNhbiBhIHBlcnNvbiBsaXZlIGEgZ29vZCBsaWZlIHdpdGhvdXQgcmVsaWdpb24/IiwKICAgICAgICAiV2hhdCBpcyB0aGUgcHVycG9zZSBvZiBodW1hbiBleGlzdGVuY2U/IiwKICAgICAgICAiU2hvdWxkIHJlbGlnaW91cyBpbnN0aXR1dGlvbnMgaGF2ZSBhIHZvaWNlIGluIG1ha2luZyBsYXdzPyIsCiAgICAgICAgIkhvdyBzaG91bGQgc29tZW9uZSBkZWNpZGUgd2hhdCBpcyByaWdodCB3aGVuIHRoZSBsYXcgYW5kIGNvbnNjaWVuY2UgZGlzYWdyZWU/IiwKICAgICAgICAiSXMgc3VmZmVyaW5nIG1lYW5pbmdmdWwgb3Igc2ltcGx5IHVuZm9ydHVuYXRlPyIsCiAgICAgICAgIlNob3VsZCBtYXJyaWFnZSBiZSBjb25zaWRlcmVkIGEgc2FjcmVkIGJvbmQgb3IgYSBjaXZpbCBjb250cmFjdD8iLAogICAgICAgICJXaGF0IHNob3VsZCBiZSB0YXVnaHQgaW4gc2Nob29scyBhYm91dCB0aGUgb3JpZ2lucyBvZiBsaWZlPyIsCiAgICAgICAgIklzIGZvcmdpdmVuZXNzIGFsd2F5cyB0aGUgcmlnaHQgcmVzcG9uc2UgdG8gYmVpbmcgd3JvbmdlZD8iLAogICAgICAgICJIb3cgc2hvdWxkIGEgY29tbXVuaXR5IHRyZWF0IHBlb3BsZSB3aG8gbGVhdmUgaXRzIGZhaXRoPyIsCiAgICAgICAgIkFyZSBzb21lIGRheXMgb3IgcGxhY2VzIGdlbnVpbmVseSBzYWNyZWQ/IiwKICAgICAgICAiV2hhdCBvYmxpZ2F0aW9ucyBkbyB0aGUgbGl2aW5nIGhhdmUgdG8gdGhlaXIgYW5jZXN0b3JzPyIsCiAgICBdLAogICAgImFnZSI6IFsKICAgICAgICAiSG93IHNob3VsZCB3b3JrcGxhY2VzIGJhbGFuY2UgZXhwZXJpZW5jZSBhZ2FpbnN0IGZyZXNoIGlkZWFzPyIsCiAgICAgICAgIklzIHNvY2lhbCBtZWRpYSBnb29kIG9yIGJhZCBmb3Igc29jaWV0eT8iLAogICAgICAgICJTaG91bGQgcmV0aXJlbWVudCBhZ2UgYmUgcmFpc2VkIGFzIHBlb3BsZSBsaXZlIGxvbmdlcj8iLAogICAgICAgICJIb3cgbXVjaCBzaG91bGQgeW91bmcgcGVvcGxlIGRlZmVyIHRvIHRoZWlyIGVsZGVycz8iLAogICAgICAgICJJcyByZW1vdGUgd29yayBiZXR0ZXIgdGhhbiB3b3JraW5nIGluIGFuIG9mZmljZT8iLAogICAgICAgICJIb3cgc2hvdWxkIHNvY2lldGllcyBhZGRyZXNzIGNsaW1hdGUgY2hhbmdlIGNvc3RzIGFjcm9zcyBnZW5lcmF0aW9ucz8iLAogICAgICAgICJTaG91bGQgdW5pdmVyc2l0eSBlZHVjYXRpb24gYmUgZnJlZSBmb3IgZXZlcnlvbmU/IiwKICAgICAgICAiSG93IGhhcyB0ZWNobm9sb2d5IGNoYW5nZWQgd2hhdCBpdCBtZWFucyB0byBoYXZlIGEgZ29vZCBsaWZlPyIsCiAgICAgICAgIkFyZSB0cmFkaXRpb25hbCBjYXJlZXIgcGF0aHMgc3RpbGwgd29ydGggZm9sbG93aW5nPyIsCiAgICAgICAgIkhvdyBzaG91bGQgZmFtaWxpZXMgZGVjaWRlIHdoZXJlIGVsZGVybHkgcmVsYXRpdmVzIGxpdmU/IiwKICAgICAgICAiSXMgaXQgYmV0dGVyIHRvIHNhdmUgbW9uZXkgb3Igc3BlbmQgaXQgb24gZXhwZXJpZW5jZXM/IiwKICAgICAgICAiU2hvdWxkIG9sZGVyIHdvcmtlcnMgYmUgcmV0cmFpbmVkIG9yIHJldGlyZWQgZWFybHk/IiwKICAgICAgICAiSG93IHNob3VsZCBjb21tdW5pdGllcyBwYXNzIGtub3dsZWRnZSBiZXR3ZWVuIGdlbmVyYXRpb25zPyIsCiAgICAgICAgIklzIGhvbWUgb3duZXJzaGlwIHN0aWxsIGEgcmVhbGlzdGljIGdvYWw/IiwKICAgICAgICAiV2hhdCByZXNwb25zaWJpbGl0aWVzIGRvIG9sZGVyIGdlbmVyYXRpb25zIG93ZSB0byB5b3VuZ2VyIG9uZXM/IiwKICAgIF0sCn0KCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgRXh0ZXJuYWwgYmVuY2htYXJrcwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgpkZWYgbG9hZF9nbG9iYWxfb3Bpbmlvbl9xdWVzdGlvbnMoKSAtPiBsaXN0OgogICAgIiIiQ3Jvc3MtY291bnRyeSBvcGluaW9uIGl0ZW1zLCBpZiByZWFjaGFibGUuCgogICAgQW50aHJvcGljJ3MgbGxtX2dsb2JhbF9vcGluaW9ucyBpcyBkcmF3biBmcm9tIHRoZSBXb3JsZCBWYWx1ZXMgU3VydmV5CiAgICBhbmQgdGhlIFBldyBHbG9iYWwgQXR0aXR1ZGVzIHN1cnZleSwgc28gdGhlIHF1ZXN0aW9ucyBhcmUgYWxyZWFkeSBrbm93bgogICAgdG8gc3BsaXQgYWxvbmcgdGhlIGxpbmVzIHRoaXMgcHJvamVjdCBzdGVlcnMgb24uCiAgICAiIiIKICAgIHRyeToKICAgICAgICBmcm9tIGRhdGFzZXRzIGltcG9ydCBsb2FkX2RhdGFzZXQKCiAgICAgICAgZGF0YXNldCA9IGxvYWRfZGF0YXNldCgiQW50aHJvcGljL2xsbV9nbG9iYWxfb3BpbmlvbnMiLCBzcGxpdD0idHJhaW4iKQogICAgICAgIHF1ZXN0aW9ucyA9IFtdCiAgICAgICAgZm9yIHJvdyBpbiBkYXRhc2V0OgogICAgICAgICAgICBxdWVzdGlvbiA9IChyb3cuZ2V0KCJxdWVzdGlvbiIpIG9yICIiKS5zdHJpcCgpCiAgICAgICAgICAgIGlmIDIwIDwgbGVuKHF1ZXN0aW9uKSA8IDQwMDoKICAgICAgICAgICAgICAgIHF1ZXN0aW9ucy5hcHBlbmQocXVlc3Rpb24pCiAgICAgICAgcHJpbnQoZiJnbG9iYWwgb3BpbmlvbnMgKFdWUy9QZXcpOiB7bGVuKHF1ZXN0aW9ucyk6LH0gcXVlc3Rpb25zIikKICAgICAgICByZXR1cm4gcXVlc3Rpb25zCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBwcmludChmImNvdWxkIG5vdCBsb2FkIGdsb2JhbCBvcGluaW9ucyAoe2V4Y30pOyBjdXJhdGVkIGJhbmsgb25seSIpCiAgICAgICAgcmV0dXJuIFtdCgoKZGVmIGxvYWRfYW50aHJvcGljX3BlcnNvbmFfcXVlc3Rpb25zKGxpbWl0OiBpbnQgPSAyMDAwKSAtPiBsaXN0OgogICAgIiIiQW50aHJvcGljIHBlcnNvbmEgZXZhbCBzdGF0ZW1lbnRzLCBpZiByZWFjaGFibGUuIiIiCiAgICB0cnk6CiAgICAgICAgZnJvbSBkYXRhc2V0cyBpbXBvcnQgbG9hZF9kYXRhc2V0CgogICAgICAgIGRhdGFzZXQgPSBsb2FkX2RhdGFzZXQoCiAgICAgICAgICAgICJBbnRocm9waWMvbW9kZWwtd3JpdHRlbi1ldmFscyIsIGRhdGFfZmlsZXM9InBlcnNvbmEvKi5qc29ubCIsIHNwbGl0PSJ0cmFpbiIKICAgICAgICApCiAgICAgICAgcXVlc3Rpb25zID0gW10KICAgICAgICBmb3Igcm93IGluIGRhdGFzZXQ6CiAgICAgICAgICAgIHF1ZXN0aW9uID0gKHJvdy5nZXQoInF1ZXN0aW9uIikgb3Igcm93LmdldCgic3RhdGVtZW50Iikgb3IgIiIpLnN0cmlwKCkKICAgICAgICAgICAgaWYgMjAgPCBsZW4ocXVlc3Rpb24pIDwgNDAwOgogICAgICAgICAgICAgICAgcXVlc3Rpb25zLmFwcGVuZChxdWVzdGlvbikKICAgICAgICAgICAgaWYgbGVuKHF1ZXN0aW9ucykgPj0gbGltaXQ6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIHByaW50KGYiQW50aHJvcGljIHBlcnNvbmE6IHtsZW4ocXVlc3Rpb25zKTosfSBxdWVzdGlvbnMiKQogICAgICAgIHJldHVybiBxdWVzdGlvbnMKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIHByaW50KGYiY291bGQgbm90IGxvYWQgQW50aHJvcGljIHBlcnNvbmEgZXZhbHMgKHtleGN9KTsgY3VyYXRlZCBiYW5rIG9ubHkiKQogICAgICAgIHJldHVybiBbXQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBMZWFrYWdlIGZpbHRlcgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgpkZWYgZmlsdGVyX2xlYWthZ2UoY2FuZGlkYXRlczogbGlzdCwgY2FsaWJyYXRpb25fcHJvbXB0czogbGlzdCwgdGhyZXNob2xkOiBmbG9hdCB8IE5vbmUgPSBOb25lKSAtPiB0dXBsZToKICAgICIiIkRyb3AgY2FuZGlkYXRlcyB0b28gc2ltaWxhciB0byBhbnkgY2FsaWJyYXRpb24gcHJvbXB0LgoKICAgIFJldHVybnMgKGtlcHQsIGRyb3BwZWQpLiBTaW1pbGFyaXR5IGlzIHNlbnRlbmNlLWVtYmVkZGluZyBjb3NpbmUsIHNvCiAgICB0aGlzIGNhdGNoZXMgcGFyYXBocmFzZXMgdGhhdCBleGFjdCB0ZXh0IG1hdGNoaW5nIHdvdWxkIG1pc3MuCiAgICAiIiIKICAgIHRocmVzaG9sZCA9IGNmZy5MRUFLQUdFX1RIUkVTSE9MRCBpZiB0aHJlc2hvbGQgaXMgTm9uZSBlbHNlIHRocmVzaG9sZAogICAgaWYgbm90IGNhbmRpZGF0ZXMgb3Igbm90IGNhbGlicmF0aW9uX3Byb21wdHM6CiAgICAgICAgcmV0dXJuIGNhbmRpZGF0ZXMsIFtdCgogICAgZnJvbSBzZW50ZW5jZV90cmFuc2Zvcm1lcnMgaW1wb3J0IFNlbnRlbmNlVHJhbnNmb3JtZXIsIHV0aWwKCiAgICBlbmNvZGVyID0gU2VudGVuY2VUcmFuc2Zvcm1lcihjZmcuTEVBS0FHRV9NT0RFTCkKICAgIGNhbmRpZGF0ZV9lbWJlZGRpbmdzID0gZW5jb2Rlci5lbmNvZGUoY2FuZGlkYXRlcywgY29udmVydF90b190ZW5zb3I9VHJ1ZSwgc2hvd19wcm9ncmVzc19iYXI9RmFsc2UpCiAgICBjYWxpYnJhdGlvbl9lbWJlZGRpbmdzID0gZW5jb2Rlci5lbmNvZGUoY2FsaWJyYXRpb25fcHJvbXB0cywgY29udmVydF90b190ZW5zb3I9VHJ1ZSwgc2hvd19wcm9ncmVzc19iYXI9RmFsc2UpCgogICAgc2ltaWxhcml0eSA9IHV0aWwuY29zX3NpbShjYW5kaWRhdGVfZW1iZWRkaW5ncywgY2FsaWJyYXRpb25fZW1iZWRkaW5ncykKICAgIG1heF9zaW1pbGFyaXR5ID0gc2ltaWxhcml0eS5tYXgoZGltPTEpLnZhbHVlcy5jcHUoKS5udW1weSgpCgogICAga2VwdCA9IFtjYW5kaWRhdGUgZm9yIGNhbmRpZGF0ZSwgcyBpbiB6aXAoY2FuZGlkYXRlcywgbWF4X3NpbWlsYXJpdHkpIGlmIHMgPCB0aHJlc2hvbGRdCiAgICBkcm9wcGVkID0gW2NhbmRpZGF0ZSBmb3IgY2FuZGlkYXRlLCBzIGluIHppcChjYW5kaWRhdGVzLCBtYXhfc2ltaWxhcml0eSkgaWYgcyA+PSB0aHJlc2hvbGRdCiAgICBwcmludChmImxlYWthZ2UgZmlsdGVyIGF0IHt0aHJlc2hvbGR9OiBrZXB0IHtsZW4oa2VwdCk6LH0sIGRyb3BwZWQge2xlbihkcm9wcGVkKTosfSIpCiAgICByZXR1cm4ga2VwdCwgZHJvcHBlZAoKCmRlZiBidWlsZF9ldmFsX3NldHMocGVyX2F4aXM6IGludCB8IE5vbmUgPSBOb25lLCBzYXZlOiBib29sID0gVHJ1ZSkgLT4gZGljdDoKICAgICIiIkJ1aWxkIG9uZSBjbGVhbiwgbGVha2FnZS1maWx0ZXJlZCBxdWVzdGlvbiBzZXQgcGVyIGF4aXMuIiIiCiAgICBwZXJfYXhpcyA9IHBlcl9heGlzIG9yIGNmZy5FVkFMX1FVRVNUSU9OU19QRVJfQVhJUwoKICAgIGV4dGVybmFsX3F1ZXN0aW9ucyA9IGxvYWRfZ2xvYmFsX29waW5pb25fcXVlc3Rpb25zKCkgKyBsb2FkX2FudGhyb3BpY19wZXJzb25hX3F1ZXN0aW9ucygpCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoY2ZnLlNFRUQpCgogICAgZXZhbF9zZXRzID0ge30KICAgIGZvciBheGlzIGluIGNmZy5BWEVTOgogICAgICAgIHRyeToKICAgICAgICAgICAgY2FsaWJyYXRpb25fcHJvbXB0cyA9IGxvYWRfcGFpcnMoYXhpcylbInByb21wdCJdLmRyb3BuYSgpLnVuaXF1ZSgpLnRvbGlzdCgpCiAgICAgICAgZXhjZXB0IEZpbGVOb3RGb3VuZEVycm9yOgogICAgICAgICAgICBjYWxpYnJhdGlvbl9wcm9tcHRzID0gW10KICAgICAgICAgICAgcHJpbnQoZiJ7YXhpc306IG5vIGNhbGlicmF0aW9uIHBhaXJzIGZvdW5kLCBza2lwcGluZyBsZWFrYWdlIGZpbHRlciIpCgogICAgICAgIHBvb2wgPSBsaXN0KENVUkFURURfUVVFU1RJT05TW2F4aXNdKQogICAgICAgIGlmIGV4dGVybmFsX3F1ZXN0aW9uczoKICAgICAgICAgICAgZXh0cmEgPSBsaXN0KGV4dGVybmFsX3F1ZXN0aW9ucykKICAgICAgICAgICAgcm5nLnNodWZmbGUoZXh0cmEpCiAgICAgICAgICAgIHBvb2wgPSBwb29sICsgZXh0cmFbOiBwZXJfYXhpcyAqIDRdCgogICAgICAgIGlmIGNhbGlicmF0aW9uX3Byb21wdHM6CiAgICAgICAgICAgIHBvb2wsIF8gPSBmaWx0ZXJfbGVha2FnZShwb29sLCBjYWxpYnJhdGlvbl9wcm9tcHRzKQoKICAgICAgICAjIEtlZXAgdGhlIGN1cmF0ZWQgYmFuayBmaXJzdCwgdGhlbiB0b3AgdXAgZnJvbSB0aGUgZmlsdGVyZWQgcG9vbC4KICAgICAgICBzZWVuLCBxdWVzdGlvbnMgPSBzZXQoKSwgW10KICAgICAgICBmb3IgcXVlc3Rpb24gaW4gcG9vbDoKICAgICAgICAgICAga2V5ID0gcXVlc3Rpb24ubG93ZXIoKS5zdHJpcCgpCiAgICAgICAgICAgIGlmIGtleSBub3QgaW4gc2VlbjoKICAgICAgICAgICAgICAgIHNlZW4uYWRkKGtleSkKICAgICAgICAgICAgICAgIHF1ZXN0aW9ucy5hcHBlbmQocXVlc3Rpb24pCiAgICAgICAgICAgIGlmIGxlbihxdWVzdGlvbnMpID49IHBlcl9heGlzOgogICAgICAgICAgICAgICAgYnJlYWsKCiAgICAgICAgZXZhbF9zZXRzW2F4aXNdID0gcXVlc3Rpb25zCiAgICAgICAgcHJpbnQoZiJ7YXhpczo5c30gZXZhbCBxdWVzdGlvbnM6IHtsZW4ocXVlc3Rpb25zKTosfSIpCgogICAgICAgIGlmIHNhdmU6CiAgICAgICAgICAgIHBhdGggPSBjZmcuQkVOQ0hNQVJLX0RJUiAvIGYiZXZhbF96ZXJvX3Nob3Rfe2F4aXN9Lmpzb24iCiAgICAgICAgICAgIHdpdGggb3BlbihwYXRoLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGZoOgogICAgICAgICAgICAgICAganNvbi5kdW1wKHsiYXhpcyI6IGF4aXMsICJuIjogbGVuKHF1ZXN0aW9ucyksICJxdWVzdGlvbnMiOiBxdWVzdGlvbnN9LCBmaCwgaW5kZW50PTIpCiAgICAgICAgICAgIHByaW50KGYiICBzYXZlZCB7cGF0aC5uYW1lfSIpCgogICAgcmV0dXJuIGV2YWxfc2V0cwoKCmRlZiBsb2FkX2V2YWxfc2V0KGF4aXM6IHN0cikgLT4gbGlzdDoKICAgIHBhdGggPSBjZmcuQkVOQ0hNQVJLX0RJUiAvIGYiZXZhbF96ZXJvX3Nob3Rfe2F4aXN9Lmpzb24iCiAgICBpZiBub3QgcGF0aC5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIntwYXRofSBtaXNzaW5nLiBSdW4gYnVpbGRfZXZhbF9zZXRzKCkgZmlyc3QuIikKICAgIHdpdGggb3BlbihwYXRoLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGZoOgogICAgICAgIHJldHVybiBqc29uLmxvYWQoZmgpWyJxdWVzdGlvbnMiXQo=',
    'config.py': 'IiIiU2V0dGluZ3MgZm9yIHRoZSBNSU9SUEEgcGx1cmFsaXN0aWMgc3RlZXJpbmcgcGlwZWxpbmUuCgpFdmVyeXRoaW5nIHR1bmFibGUgbGl2ZXMgaGVyZSBzbyB0aGUgbm90ZWJvb2sgc3RheXMgdGhpbiBhbmQgZXZlcnkgc3RhZ2UKcmVhZHMgdGhlIHNhbWUgdmFsdWVzLgoiIiIKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKZGVmIF9kZWZhdWx0X21vZGVsX2R0eXBlKCk6CiAgICAiIiJmcDMyLCBub3QgZnAxNi4KCiAgICBmcDE2IHRvcHMgb3V0IGFyb3VuZCA2NSw1MDQuIFNtYWxsIG1vZGVscyBzb21ldGltZXMgcHJvZHVjZSBhY3RpdmF0aW9uCiAgICBzcGlrZXMgaW4gYSBmZXcgaGlkZGVuIGRpbWVuc2lvbnMgdGhhdCBnbyBwYXN0IHRoYXQsIHdoaWNoIHR1cm5zIGludG8KICAgIGluZi4gc2tsZWFybidzIFBDQSBhbmQgbG9naXN0aWMgcmVncmVzc2lvbiBjcmFzaCBvbiBpbmYsIGFuZCBOYU4gbG9naXRzCiAgICBkdXJpbmcgc3RlZXJlZCBnZW5lcmF0aW9uIHRyaWdnZXIgYSBDVURBIGRldmljZS1zaWRlIGFzc2VydC4gZnAzMiBoYXMKICAgIGVub3VnaCBoZWFkcm9vbSB0aGF0IHRoaXMgZG9lc24ndCBoYXBwZW4sIGFuZCBpdCBjb3N0cyBhbG1vc3Qgbm90aGluZwogICAgZXh0cmEgZm9yIG1vZGVscyB0aGlzIHNpemUgb24gYSBub3JtYWwgR1BVLgogICAgIiIiCiAgICBpbXBvcnQgdG9yY2gKCiAgICByZXR1cm4gdG9yY2guZmxvYXQzMgoKCk1PREVMX0RUWVBFID0gX2RlZmF1bHRfbW9kZWxfZHR5cGUoKQoKCmRlZiBfZW5hYmxlX3RmMzIoKToKICAgICIiIkxldCBtYXRtdWxzIG9uIEFtcGVyZSsgR1BVcyB1c2UgVEYzMiBpbnN0ZWFkIG9mIGZ1bGwgZnAzMi4KCiAgICBURjMyIGtlZXBzIGZwMzIncyBleHBvbmVudCByYW5nZSwgc28gbm9uZSBvZiB0aGUgb3ZlcmZsb3cgYmVoYXZpb3VyCiAgICBfZGVmYXVsdF9tb2RlbF9kdHlwZSgpIGFib3ZlIGlzIGd1YXJkaW5nIGFnYWluc3QgY29tZXMgYmFjay4gSXQgb25seQogICAgdHJpbXMgbWFudGlzc2EgYml0cywgd2hpY2ggY3V0cyBtYXRtdWwgdGltZSBzdWJzdGFudGlhbGx5IG9uIEFtcGVyZSBhbmQKICAgIG5ld2VyIGNhcmRzIGFuZCBjb3N0cyBub3RoaW5nIG9uIEdQVXMgdGhhdCBkb24ndCBzdXBwb3J0IGl0LgogICAgIiIiCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoCgogICAgICAgIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgIHRvcmNoLmJhY2tlbmRzLmN1ZGEubWF0bXVsLmFsbG93X3RmMzIgPSBUcnVlCiAgICAgICAgICAgIHRvcmNoLmJhY2tlbmRzLmN1ZG5uLmFsbG93X3RmMzIgPSBUcnVlCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKCgpfZW5hYmxlX3RmMzIoKQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBQYXRocwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudC5wYXJlbnQKCkRBVEFTRVRfRElSID0gUk9PVCAvICJkYXRhc2V0IgpTVVJWRVlfUEFUSCA9IERBVEFTRVRfRElSIC8gInN1cnZleS5qc29ubCIKQ09OVkVSU0FUSU9OU19QQVRIID0gREFUQVNFVF9ESVIgLyAiY29udmVyc2F0aW9ucy5qc29ubCIKVVRURVJBTkNFU19QQVRIID0gREFUQVNFVF9ESVIgLyAidXR0ZXJhbmNlcy5qc29ubCIKClJFU1VMVFNfRElSID0gUk9PVCAvICJyZXN1bHRzIgpQQUlSU19ESVIgPSBSRVNVTFRTX0RJUiAvICJwYWlycyIKVkVDVE9SU19ESVIgPSBSRVNVTFRTX0RJUiAvICJ2ZWN0b3JzIgpHRU5FUkFUSU9OU19ESVIgPSBSRVNVTFRTX0RJUiAvICJnZW5lcmF0aW9ucyIKRVZBTF9ESVIgPSBSRVNVTFRTX0RJUiAvICJldmFsdWF0aW9uIgpCRU5DSE1BUktfRElSID0gUkVTVUxUU19ESVIgLyAiYmVuY2htYXJrcyIKCmZvciBvdXRwdXRfZGlyIGluIChSRVNVTFRTX0RJUiwgUEFJUlNfRElSLCBWRUNUT1JTX0RJUiwgR0VORVJBVElPTlNfRElSLCBFVkFMX0RJUiwgQkVOQ0hNQVJLX0RJUik6CiAgICBvdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgRGVtb2dyYXBoaWMgYXhlcwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgT3JpZ2luICAgLT4gV2VzdGVybiB2cyBOb24tV2VzdGVybgojIFJlbGlnaW9uIC0+IFJlbGlnaW91cyB2cyBTZWN1bGFyCiMgQWdlICAgICAgLT4gWW91bmcgKEdlbiBZL1opIHZzIE9sZCAoR2VuIFgvQm9vbWVycykKCldFU1RFUk5fUkVHSU9OUyA9IHsiRXVyb3BlIiwgIkFtZXJpY2FzIiwgIk9jZWFuaWEifQpOT05fV0VTVEVSTl9SRUdJT05TID0geyJBZnJpY2EiLCAiQXNpYSJ9CgojIENvdW50cyBpbiBkYXRhc2V0L3N1cnZleS5qc29ubCwgcmVsaWdpb24uc2ltcGxpZmllZDoKIyBObyBBZmZpbGlhdGlvbiA4NTEsIENocmlzdGlhbiA0ODcsIFByZWZlciBub3QgdG8gc2F5IDU5LCBKZXdpc2ggNDIsCiMgTXVzbGltIDMxLCBPdGhlciAzMC4gIlByZWZlciBub3QgdG8gc2F5IiBzaXRzIGluIG5laXRoZXIgc2V0IG9uIHB1cnBvc2UsCiMgdGhvc2UgdXNlcnMgYXJlIGRyb3BwZWQgZnJvbSB0aGlzIGF4aXMgcmF0aGVyIHRoYW4gZ3Vlc3NlZCBhdC4KUkVMSUdJT1VTX0xBQkVMUyA9IHsiQ2hyaXN0aWFuIiwgIk11c2xpbSIsICJKZXdpc2giLCAiSGluZHUiLCAiQnVkZGhpc3QiLCAiT3RoZXIifQpTRUNVTEFSX0xBQkVMUyA9IHsiTm8gQWZmaWxpYXRpb24ifQoKWU9VTkdfQUdFX0JSQUNLRVRTID0geyIxOC0yNCB5ZWFycyBvbGQiLCAiMjUtMzQgeWVhcnMgb2xkIiwgIjM1LTQ0IHllYXJzIG9sZCJ9Ck9MRF9BR0VfQlJBQ0tFVFMgPSB7IjQ1LTU0IHllYXJzIG9sZCIsICI1NS02NCB5ZWFycyBvbGQiLCAiNjUrIHllYXJzIG9sZCJ9CgpBWEVTID0gKCJvcmlnaW4iLCAicmVsaWdpb24iLCAiYWdlIikKCkFYSVNfUE9MRVMgPSB7CiAgICAib3JpZ2luIjogKCJ3ZXN0ZXJuIiwgIm5vbl93ZXN0ZXJuIiksCiAgICAicmVsaWdpb24iOiAoInJlbGlnaW91cyIsICJzZWN1bGFyIiksCiAgICAiYWdlIjogKCJ5b3VuZyIsICJvbGQiKSwKfQoKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBDYWxpYnJhdGlvbiBwYWlycwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCgojIENvc2luZSB0aHJlc2hvbGQgZm9yIG1hdGNoaW5nIHF1ZXN0aW9ucyBieSBtZWFuaW5nIG9uY2UgZXhhY3QgdGV4dCBtYXRjaAojIHJ1bnMgb3V0LiBBZ2Ugb25seSBoYXMgMTMgZXhhY3QtbWF0Y2ggcGFpcnMsIHRvbyB0aGluIG9uIGl0cyBvd24uClNFTUFOVElDX01BVENIX1RIUkVTSE9MRCA9IDAuNzUKCiMgQSByZXNwb25zZSBjb3VudHMgYXMgbGlrZWQgYXQgb3IgYWJvdmUgdGhpcyBzY29yZS4gUFJJU00gc2NvcmVzIHJ1biAxLTEwMC4KSElHSF9SQVRJTkdfVEhSRVNIT0xEID0gNzUKCk1JTl9QQUlSU19QRVJfQVhJUyA9IDMwCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIE1vZGVscyB1bmRlciB0ZXN0CiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIE1vZGVsU3BlYzoKICAgIGhmX2lkOiBzdHIKICAgIHNob3J0X25hbWU6IHN0cgogICAgZmFtaWx5OiBzdHIKICAgIHBhcmFtczogc3RyCgogICAgQHByb3BlcnR5CiAgICBkZWYgc2x1ZyhzZWxmKSAtPiBzdHI6CiAgICAgICAgcmV0dXJuIHNlbGYuc2hvcnRfbmFtZS5yZXBsYWNlKCIvIiwgIl8iKS5yZXBsYWNlKCIuIiwgIi0iKQoKCk1PREVMUyA9IFsKICAgIE1vZGVsU3BlYygiSHVnZ2luZ0ZhY2VUQi9TbW9sTE0yLTEzNU0tSW5zdHJ1Y3QiLCAiU21vbExNMi0xMzVNIiwgInNtb2xsbTIiLCAiMTM1TSIpLAogICAgTW9kZWxTcGVjKCJIdWdnaW5nRmFjZVRCL1Ntb2xMTTItMzYwTS1JbnN0cnVjdCIsICJTbW9sTE0yLTM2ME0iLCAic21vbGxtMiIsICIzNjBNIiksCiAgICBNb2RlbFNwZWMoIkh1Z2dpbmdGYWNlVEIvU21vbExNMi0xLjdCLUluc3RydWN0IiwgIlNtb2xMTTItMS43QiIsICJzbW9sbG0yIiwgIjEuN0IiKSwKICAgIE1vZGVsU3BlYygiUXdlbi9Rd2VuMi41LTAuNUItSW5zdHJ1Y3QiLCAiUXdlbjIuNS0wLjVCIiwgInF3ZW4yLjUiLCAiMC41QiIpLAogICAgTW9kZWxTcGVjKCJRd2VuL1F3ZW4yLjUtMS41Qi1JbnN0cnVjdCIsICJRd2VuMi41LTEuNUIiLCAicXdlbjIuNSIsICIxLjVCIiksCiAgICBNb2RlbFNwZWMoIlF3ZW4vUXdlbjIuNS0zQi1JbnN0cnVjdCIsICJRd2VuMi41LTNCIiwgInF3ZW4yLjUiLCAiM0IiKSwKXQoKTU9ERUxTX0JZX05BTUUgPSB7bS5zaG9ydF9uYW1lOiBtIGZvciBtIGluIE1PREVMU30KCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgU3RlZXJpbmcKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQoKVkVDVE9SX01FVEhPRFMgPSAoIk1vRCIsICJQb0QiLCAiUG9FIiwgIkNvRSIpCgojIEluamVjdGlvbiBkZXB0aCBhcyBhIGZyYWN0aW9uIG9mIHRvdGFsIGxheWVycywgc28gaXQgY2FycmllcyBvdmVyIHRvIG1vZGVscwojIHdpdGggZGlmZmVyZW50IGRlcHRocy4gSW0gJiBMaSAoMjAyNikgc3RlZXIgTGxhbWEtMi03Yi1jaGF0IGF0IGxheWVyIDEzIG9mCiMgMzIsIHdoaWNoIGlzIGFib3V0IDAuNDEuIFRoZSBsYXllciBzd2VlcCBjaGVja3MgaWYgdGhhdCBob2xkcyBoZXJlIHRvby4KTEFZRVJfREVQVEhfRlJBQ1RJT04gPSAwLjQ1CgojIE11bHRpcGxpZXIgc2V0IG1hdGNoZXMgSW0gJiBMaTogMC41IHRvIDMuMCwgcGx1cyAwIGFzIHRoZSB1bnN0ZWVyZWQgcG9pbnQuCkRFRkFVTFRfQUxQSEEgPSAxLjUKQUxQSEFfU1dFRVAgPSAoMC4wLCAwLjUsIDEuMCwgMS41LCAyLjAsIDIuNSwgMy4wKQpMQVlFUl9TV0VFUF9GUkFDVElPTlMgPSAoMC4yNSwgMC40MSwgMC41NSwgMC43MCwgMC44NSkKCiMgSG93IGFjdGl2YXRpb25zIGdldCBwb29sZWQgYmVmb3JlIGJ1aWxkaW5nIGEgdmVjdG9yLgojICAgIm1lYW4iIC0gYXZlcmFnZSBvdmVyIGV2ZXJ5IHJlYWwgdG9rZW4uIFN0YWJsZSwgZGVmYXVsdC4KIyAgICJsYXN0IiAtIGZpbmFsIHRva2VuIG9ubHksIHdoaWNoIGlzIHdoYXQgSW0gJiBMaSB1c2UuCkFDVElWQVRJT05fUE9PTElORyA9ICJtZWFuIgoKIyBSZXNjYWxlIFBvRCwgUG9FIGFuZCBDb0UgdG8gdGhlIHNhbWUgbm9ybSBhcyBNb0QgYmVmb3JlIHN0ZWVyaW5nLgojCiMgSW0gJiBMaSBkZWZpbmUgUG9EIGFuZCBQb0UgYXQgdW5pdCBsZW5ndGggYW5kIENvRSBhdCB0aGUgc3RhbmRhcmQKIyBkZXZpYXRpb24gYWxvbmcgaXRzIG93biBkaXJlY3Rpb24sIHRoZW4gc2VhcmNoIGEgc2VwYXJhdGUgbXVsdGlwbGllciBwZXIKIyBtZXRob2QgdG8gbWFrZSB0aGVtIGNvbXBhcmFibGUuIFJlc2NhbGluZyB0byB8TW9EfCBkb2VzIHRoZSBzYW1lIGpvYgojIGRpcmVjdGx5LCBzbyBvbmUgYWxwaGEgbWVhbnMgdGhlIHNhbWUgcHVzaCBmb3IgZXZlcnkgbWV0aG9kLgojCiMgU2V0IEZhbHNlIHRvIHVzZSB0aGUgcGFwZXIncyBvd24gc2NhbGluZyBpbnN0ZWFkLiBUaGVpciByZXN1bHQgdGhhdCBDb0UKIyB1bmRlcnBlcmZvcm1zIGlzIHBhcnRseSBiZWNhdXNlIGl0cyBuYXR1cmFsIG1hZ25pdHVkZSBpcyBzbWFsbCwgc28gdGhpcwojIGZsYWcgY2hhbmdlcyBob3cgdGhlIGZvdXIgbWV0aG9kcyByYW5rIGFnYWluc3QgZWFjaCBvdGhlci4KTk9STUFMSVNFX1RPX01PRF9OT1JNID0gVHJ1ZQoKIyBFcXVhbGlzaW5nIHRoZSBwdXNoIGFjcm9zcyBheGVzIGFuZCBtb2RlbHMuCiMKIyBhbHBoYSBvbmx5IG11bHRpcGxpZXMgdGhlIHZlY3RvcjsgaXQgZG9lcyBub3Qgc2V0IHRoZSBzdHJlbmd0aCBvbiBpdHMgb3duLgojIEVhY2ggYXhpcydzIHZlY3RvciBjYXJyaWVzIHdoYXRldmVyIGxlbmd0aCBpdHMgb3duIGFjdGl2YXRpb25zIHByb2R1Y2VkLCBzbwojIG9uZSBzaGFyZWQgYWxwaGEgcHVzaGVzIHJlbGlnaW9uIHNldmVyYWwgdGltZXMgaGFyZGVyIHRoYW4gb3JpZ2luIGFuZCB0aGUKIyBheGVzIGFyZSBuZXZlciBjb21wYXJlZCBhdCB0aGUgc2FtZSBzdHJlbmd0aC4KIwojIFRoZSBmaXJzdCBhdHRlbXB0IGF0IGZpeGluZyB0aGlzIGVxdWFsaXNlZCB0byBhbiBhYnNvbHV0ZSBub3JtLCB3aGljaCB3YXMKIyBzdGlsbCB3cm9uZzogYSBwdXNoIG9mIDIxIGlzIDQuMyUgb2YgU21vbExNMi0zNjBNJ3MgaGlkZGVuIHN0YXRlIGJ1dCA4MiUgb2YKIyBRd2VuMi41LTAuNUIncywgd2hvc2UgcmVwcmVzZW50YXRpb25zIGFyZSBmYXIgc21hbGxlci4gVGhlIHNjYWxlLWZyZWUKIyBxdWFudGl0eSBpcyB8YWxwaGEgKiB2fCAvIHxofCwgc28gdGhhdCBpcyB3aGF0IGdldHMgaGVsZCBjb25zdGFudCBub3cuCiMKIyAwLjI1IGNvbWVzIGZyb20gdGhlIG1lYXN1cmVkIGRhdGEuIEFjcm9zcyBzaXggbW9kZWxzIHRoZSBheGVzIGxhbmQgaW4KIyBub24tb3ZlcmxhcHBpbmcgYmFuZHMgb2Ygcm91Z2hseSAxMiUgKG9yaWdpbiksIDI1JSAoYWdlKSBhbmQgNTklIChyZWxpZ2lvbiksCiMgYW5kIHRoZSBzdWNjZXNzZnVsIGNvbmRpdGlvbnMgY2x1c3RlciBpbiB0aGUgbWlkZGxlIGJhbmQuIE9yaWdpbiBhdCAxMiUgbG9va3MKIyB1bmRlci1wdXNoZWQgYW5kIHJlbGlnaW9uIGF0IDU5JSBpcyBvdmVyd3JpdHRlbiByYXRoZXIgdGhhbiBudWRnZWQuCiMKIyBXaXRoIFJ1bkNvbmZpZy5lcXVhbGlzZV9wdXNoIHNldCwgYWxwaGEgaXMgcmVzY2FsZWQgcGVyIGF4aXMgYW5kIHBlciBtb2RlbCBzbwojIGV2ZXJ5IGNvbmRpdGlvbiByZWNlaXZlcyBQVVNIX1RBUkdFVF9SQVRJTyBvZiBpdHMgb3duIGhpZGRlbi1zdGF0ZSBub3JtLgojIGFscGhhIHN0aWxsIHNjYWxlcyB0aGUgdGFyZ2V0IHJlbGF0aXZlIHRvIERFRkFVTFRfQUxQSEEsIHNvIGEgc3dlZXAgd29ya3MuClBVU0hfVEFSR0VUX1JBVElPID0gMC4yNQoKIyBTdGVlciBvbmx5IHRoZSB0b2tlbnMgYmVpbmcgZ2VuZXJhdGVkLCBuZXZlciB0aGUgcHJvbXB0IGl0c2VsZi4KU1RFRVJfUFJPTVBUX1RPS0VOUyA9IEZhbHNlCgpNQVhfTkVXX1RPS0VOUyA9IDEyOAoKCmRlZiBfZGVmYXVsdF9nZW5lcmF0aW9uX2JhdGNoX3NpemUoKSAtPiBpbnQ6CiAgICAiIiJQaWNrIGEgYmF0Y2ggc2l6ZSBmcm9tIHdoYXRldmVyIEdQVSBpcyBhY3R1YWxseSBhdHRhY2hlZC4KCiAgICBDb2xhYiBoYW5kcyBvdXQgYSBkaWZmZXJlbnQgY2FyZCBkZXBlbmRpbmcgb24gYXZhaWxhYmlsaXR5LCBzbyB0aGlzCiAgICByZWFkcyB0aGUgcmVhbCBtZW1vcnkgaW5zdGVhZCBvZiBhc3N1bWluZyBvbmUgY2FyZCBldmVyeSB0aW1lLgogICAgIiIiCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoCgogICAgICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByZXR1cm4gMTYKICAgICAgICB0b3RhbF9nYiA9IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9wcm9wZXJ0aWVzKDApLnRvdGFsX21lbW9yeSAvIDFlOQogICAgICAgIGlmIHRvdGFsX2diID49IDcwOiAgICAgICMgQTEwMCA4MEdCLCBIMTAwCiAgICAgICAgICAgIHJldHVybiA5NgogICAgICAgIGlmIHRvdGFsX2diID49IDM1OiAgICAgICMgQTEwMCA0MEdCCiAgICAgICAgICAgIHJldHVybiA2NAogICAgICAgIGlmIHRvdGFsX2diID49IDIwOiAgICAgICMgVjEwMCAzMkdCCiAgICAgICAgICAgIHJldHVybiAzMgogICAgICAgIHJldHVybiAxNiAgICAgICAgICAgICAgICAjIFQ0IDE2R0Igb3IgdW5rbm93bgogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gMTYKCgpHRU5FUkFUSU9OX0JBVENIX1NJWkUgPSBfZGVmYXVsdF9nZW5lcmF0aW9uX2JhdGNoX3NpemUoKQpURU1QRVJBVFVSRSA9IDAuNwpUT1BfUCA9IDAuOQpTRUVEID0gMjAyNjA4MDIKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgRXZhbHVhdGlvbgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgVHdvIGZ1bGx5LWF1dG9tYXRpYyBtZXRyaWNzOgojICAgYmVydHNjb3JlICAgLSBkaWQgdGhlIGFuc3dlciBzdGF5IG9uIHRvcGljCiMgICBwZXJwbGV4aXR5ICAtIGlzIHRoZSBhbnN3ZXIgc3RpbGwgZmx1ZW50IEVuZ2xpc2gKIwojIFBsdXJhbGlzbSBpcyBzY29yZWQgc2VwYXJhdGVseSwgYnkgdGhlIHBhaXJ3aXNlIGp1ZGdlcyBpbiBqdWRnZS5weSwgb24KIyB0aGlzIHN0YWdlJ3Mgb3V0cHV0LgoKQkVSVFNDT1JFX01PREVMID0gIm1pY3Jvc29mdC9kZWJlcnRhLXhsYXJnZS1tbmxpIgpCRVJUU0NPUkVfRkFMTEJBQ0sgPSAicm9iZXJ0YS1sYXJnZSIKCiMgT25lIGZpeGVkIHNjb3JlciBmb3IgZXZlcnkgbW9kZWwsIG90aGVyd2lzZSBhIDNCIGFsd2F5cyBsb29rcyBtb3JlIGZsdWVudAojIHRoYW4gYSAxMzVNIGJ5IGNvbnN0cnVjdGlvbiBhbmQgdGhlIGNvbXBhcmlzb24gbWVhbnMgbm90aGluZy4KUEVSUExFWElUWV9TQ09SRVIgPSAiZ3B0Mi1sYXJnZSIKCiMgRHJvcCBhbnkgdGVzdCBxdWVzdGlvbiB0b28gY2xvc2UgdG8gYSBjYWxpYnJhdGlvbiBxdWVzdGlvbi4KTEVBS0FHRV9NT0RFTCA9ICJhbGwtTWluaUxNLUw2LXYyIgpMRUFLQUdFX1RIUkVTSE9MRCA9IDAuNTAKCkVWQUxfUVVFU1RJT05TX1BFUl9BWElTID0gMjAwCkNIRUNLUE9JTlRfRVZFUlkgPSAxMDAKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgUGFpcndpc2UganVkZ2luZwojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgSnVkZ2luZyBpcyBwYWlyd2lzZSAoYmFzZWxpbmUgdnMgc3RlZXJlZCBvbiB0aGUgc2FtZSBxdWVzdGlvbiksIG5vdCBhbgojIGFic29sdXRlIDEtMTAgc2NvcmUuIEFic29sdXRlIHNjb3JlcyBmcm9tIExMTSBqdWRnZXMgYnVuY2ggYXJvdW5kIDcgYW5kIDgKIyBhbmQgdGhlIGZpbmUgZGlzdGluY3Rpb25zIGFyZSBtb3N0bHkgbm9pc2U7IGEgZm9yY2VkIGNvbXBhcmlzb24gYmV0d2VlbgojIHR3byBhbnN3ZXJzIHRvIHRoZSBzYW1lIHF1ZXN0aW9uIGlzIGEgZmFyIG1vcmUgcmVsaWFibGUgaW5zdHJ1bWVudCBhbmQKIyBhbnN3ZXJzIHRoZSByZXNlYXJjaCBxdWVzdGlvbiBkaXJlY3RseS4KIwojIE5laXRoZXIganVkZ2UgY29tZXMgZnJvbSBhIGZhbWlseSB1bmRlciB0ZXN0LiBMTE0ganVkZ2VzIHJhdGUgdGhlaXIgb3duCiMgZmFtaWx5J3Mgb3V0cHV0cyBoaWdoZXIsIGFuZCBoYWxmIHRoZSB0ZXN0IHNldCBpcyBRd2VuLCBzbyBhIFF3ZW4ganVkZ2UKIyB3b3VsZCBxdWlldGx5IGFkdmFudGFnZSB0aHJlZSBvZiB0aGUgc2l4IG1vZGVscyBhbmQgY29ycnVwdCB0aGUgc2NhbGluZwojIGNvbXBhcmlzb24uIFNhbWUgcmVhc29uIHJ1bGVzIG91dCBhbnl0aGluZyBmcm9tIEh1Z2dpbmdGYWNlVEIuCgojIEhvdyBtYW55IHBhaXJzIHBlciBjb25kaXRpb24gZ28gdG8gdGhlIGp1ZGdlLiBFdmVyeSBwYWlyIGlzIGp1ZGdlZCB0d2ljZSwKIyBvbmNlIGluIGVhY2ggb3JkZXIsIHNvIHRoZSByZWFsIGNhbGwgY291bnQgaXMgZG91YmxlIHRoaXMuCiMKIyBTZXQgdG8gdGhlIGZ1bGwgcXVlc3Rpb24gc2V0IHJhdGhlciB0aGFuIGEgc2FtcGxlLiBBdCA2MCB0aGUgd2luIHJhdGVzIHdlcmUKIyByZXN0aW5nIG9uIHZlcnkgZmV3IHBhaXJzIG9uY2UgdGllcyB3ZXJlIHJlbW92ZWQgLSBRd2VuMi41LTNCIG9yaWdpbiBnYXZlIGEKIyBkZWNpc2l2ZSByYXRlIG9mIDAuNjY3IGZyb20gNiB3aW5zIGFuZCAzIGxvc3Nlcywgd2l0aCB0aGUgb3RoZXIgNTEgcGFpcnMKIyB0aWVkIGFuZCBkaXNjYXJkZWQuIE5pbmUgZGVjaXNpb25zIGNhbm5vdCBzZXBhcmF0ZSBhIHJlYWwgZWZmZWN0IGZyb20KIyBjaGFuY2UuIEp1ZGdpbmcgZXZlcnl0aGluZyByb3VnaGx5IHRyaXBsZXMgdGhlIGRlY2lkZWQgcGFpcnMgYW5kIG5hcnJvd3MKIyBldmVyeSBpbnRlcnZhbCBieSBhYm91dCBhIGZhY3RvciBvZiAxLjgsIGF0IHRoZSBjb3N0IG9mIGEgbG9uZ2VyIGp1ZGdpbmcKIyBzdGFnZS4gQ29uZmlkZW5jZSBpbnRlcnZhbHMgYXJlIHN0aWxsIHJlcG9ydGVkLCBiZWNhdXNlIGhpZ2ggdGllIHJhdGVzIG1lYW4KIyBldmVuIHRoZSBmdWxsIHNldCBsZWF2ZXMgc29tZSBjb25kaXRpb25zIHRoaW4uCkpVREdFX1BBSVJTX1BFUl9DT05ESVRJT04gPSBFVkFMX1FVRVNUSU9OU19QRVJfQVhJUwoKZGVmIF9kZWZhdWx0X2p1ZGdlX2JhdGNoX3NpemUoKSAtPiBpbnQ6CiAgICAiIiJQaWNrIGEganVkZ2UgYmF0Y2ggc2l6ZSBmcm9tIHRoZSBhdHRhY2hlZCBHUFUsIHNhbWUgaWRlYSBhcyBnZW5lcmF0aW9uLgoKICAgIEp1ZGdlIHByb21wdHMgY2FycnkgYSBmdWxsIHF1ZXN0aW9uIHBsdXMgdHdvIHJlc3BvbnNlcyBlYWNoLCBhbmQKICAgIE1pc3RyYWwtU21hbGwgaXMgMjRCLCBzbyBoZWFkcm9vbSBwZXIgZXhhbXBsZSBpcyBzbWFsbGVyIHRoYW4gZm9yIHRoZQogICAgc21hbGwgbW9kZWxzIHVuZGVyIHRlc3QgLSBzY2FsZWQgZG93biBhY2NvcmRpbmdseSBhdCBldmVyeSB0aWVyLgogICAgIiIiCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoCgogICAgICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByZXR1cm4gNAogICAgICAgIHRvdGFsX2diID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoMCkudG90YWxfbWVtb3J5IC8gMWU5CiAgICAgICAgaWYgdG90YWxfZ2IgPj0gNzA6ICAgICAgIyBBMTAwIDgwR0IsIEgxMDAKICAgICAgICAgICAgcmV0dXJuIDMyCiAgICAgICAgaWYgdG90YWxfZ2IgPj0gMzU6ICAgICAgIyBBMTAwIDQwR0IKICAgICAgICAgICAgcmV0dXJuIDE2CiAgICAgICAgaWYgdG90YWxfZ2IgPj0gMjA6ICAgICAgIyBWMTAwIDMyR0IKICAgICAgICAgICAgcmV0dXJuIDgKICAgICAgICByZXR1cm4gNCAgICAgICAgICAgICAgICAgIyBUNCAxNkdCIG9yIHVua25vd24KICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIDQKCgojIEZhbGxiYWNrIG9ubHkuIGp1ZGdlLnJ1bl9qdWRnZSgpIG5vcm1hbGx5IHNpemVzIHRoZSBiYXRjaCBmcm9tIHRoZSBtZW1vcnkKIyBhY3R1YWxseSBmcmVlIG9uY2UgYSBqdWRnZSdzIHdlaWdodHMgYXJlIHJlc2lkZW50LCBzaW5jZSBhIDdCIGFuZCBhIDI0QgojIGp1ZGdlIGhhdmUgdmVyeSBkaWZmZXJlbnQgaGVhZHJvb20gb24gdGhlIHNhbWUgY2FyZC4KSlVER0VfQkFUQ0hfU0laRSA9IF9kZWZhdWx0X2p1ZGdlX2JhdGNoX3NpemUoKQoKIyBUaGUgY2hhdCBqdWRnZXMgYW5zd2VyIHdpdGggb25lIGxpbmUgb2YgSlNPTiwgYWJvdXQgNDAgdG9rZW5zLiBBIGJhdGNoIGtlZXBzCiMgc3RlcHBpbmcgdW50aWwgZXZlcnkgc2VxdWVuY2UgaW4gaXQgaGFzIGZpbmlzaGVkLCBzbyBhIHNpbmdsZSByYW1ibGluZyByZXBseQojIGRyYWdzIHRoZSB3aG9sZSBiYXRjaCB0byB0aGUgY2FwLiAxMjggc3RpbGwgbGVhdmVzIHJvdWdobHkgdGhyZWUgdGltZXMgdGhlCiMgdHlwaWNhbCByZXBseSBsZW5ndGgsIGFuZCB0aGUgcGFyc2UtZmFpbHVyZSBjb3VudCBpcyB0aGUgY2hlY2s6IGlmIGl0IGNsaW1icwojIGFib3ZlIHRoZSB+MS0yJSBQcm9tZXRoZXVzIGFscmVhZHkgc2l0cyBhdCwgdGhpcyB3YXMgY3V0IHRvbyBmYXIuCkpVREdFX0NIQVRfTUFYX05FV19UT0tFTlMgPSAxMjgKCiMgUHJvbWV0aGV1cyB3cml0ZXMgaXRzIGZlZWRiYWNrIGJlZm9yZSB0aGUgdmVyZGljdCwgc28gaXQgZ2VudWluZWx5IG5lZWRzIHRoZQojIHJvb20uIEl0cyAxLTIlIHBhcnNlIGZhaWx1cmVzIGFyZSBhbHJlYWR5IHRydW5jYXRpb24sIHNvIGRvIG5vdCBjdXQgdGhpcy4KSlVER0VfUFJPTUVUSEVVU19NQVhfTkVXX1RPS0VOUyA9IDUxMgoKIyBQYWlycyBzZXQgYXNpZGUgZm9yIGEgaHVtYW4gdG8gc2NvcmUgbGF0ZXIsIGZvciBqdWRnZS12cy1odW1hbiBhZ3JlZW1lbnQuCkhVTUFOX1JFVklFV19TQU1QTEUgPSAxMDAKCgpAZGF0YWNsYXNzCmNsYXNzIFJ1bkNvbmZpZzoKICAgICIiIk9uZSBydW4ncyBzZXR0aW5ncywgc28gYW4gYWJsYXRpb24gZG9lc24ndCBuZWVkIHRvIGVkaXQgdGhpcyBmaWxlLiIiIgoKICAgIG1vZGVsczogbGlzdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1sYW1iZGE6IFttLnNob3J0X25hbWUgZm9yIG0gaW4gTU9ERUxTXSkKICAgIGF4ZXM6IHR1cGxlID0gQVhFUwogICAgbWV0aG9kczogdHVwbGUgPSBWRUNUT1JfTUVUSE9EUwogICAgYWxwaGFzOiB0dXBsZSA9IChERUZBVUxUX0FMUEhBLCkKICAgIGxheWVyX2ZyYWN0aW9uOiBmbG9hdCA9IExBWUVSX0RFUFRIX0ZSQUNUSU9OCiAgICBxdWVzdGlvbnNfcGVyX2F4aXM6IGludCA9IEVWQUxfUVVFU1RJT05TX1BFUl9BWElTCiAgICBpbmNsdWRlX3JhbmRvbV9jb250cm9sOiBib29sID0gVHJ1ZQogICAgIyBGYWxzZSByZXByb2R1Y2VzIHRoZSBuYWl2ZSBzZXR1cCwgb25lIGFscGhhIHNoYXJlZCBieSBldmVyeSBheGlzLgogICAgIyBUcnVlIHJlc2NhbGVzIGFscGhhIHBlciBheGlzIHNvIGV2ZXJ5IGF4aXMgcmVjZWl2ZXMgdGhlIHNhbWUgcHVzaC4KICAgIGVxdWFsaXNlX3B1c2g6IGJvb2wgPSBGYWxzZQogICAgdGFnOiBzdHIgPSAibWFpbiIKCiAgICBkZWYgZGVzY3JpYmUoc2VsZikgLT4gc3RyOgogICAgICAgIG1ldGhvZHNfcGVyX3F1ZXN0aW9uID0gbGVuKHNlbGYubWV0aG9kcykgKyAoMSBpZiBzZWxmLmluY2x1ZGVfcmFuZG9tX2NvbnRyb2wgZWxzZSAwKQogICAgICAgIHN0ZWVyZWRfdG90YWwgPSAoCiAgICAgICAgICAgIGxlbihzZWxmLm1vZGVscykgKiBsZW4oc2VsZi5heGVzKSAqIG1ldGhvZHNfcGVyX3F1ZXN0aW9uCiAgICAgICAgICAgICogbGVuKHNlbGYuYWxwaGFzKSAqIHNlbGYucXVlc3Rpb25zX3Blcl9heGlzCiAgICAgICAgKQogICAgICAgIGJhc2VsaW5lX3RvdGFsID0gbGVuKHNlbGYubW9kZWxzKSAqIGxlbihzZWxmLmF4ZXMpICogc2VsZi5xdWVzdGlvbnNfcGVyX2F4aXMKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICBmIlt7c2VsZi50YWd9XSB7bGVuKHNlbGYubW9kZWxzKX0gbW9kZWxzIHgge2xlbihzZWxmLmF4ZXMpfSBheGVzIHggIgogICAgICAgICAgICBmIntsZW4oc2VsZi5tZXRob2RzKX0gbWV0aG9kcyB4IHtsZW4oc2VsZi5hbHBoYXMpfSBhbHBoYXMgeCAiCiAgICAgICAgICAgIGYie3NlbGYucXVlc3Rpb25zX3Blcl9heGlzfSBxdWVzdGlvbnNcbiIKICAgICAgICAgICAgZiIgIHN0ZWVyZWQgZ2VuZXJhdGlvbnM6IHtzdGVlcmVkX3RvdGFsOix9XG4iCiAgICAgICAgICAgIGYiICBiYXNlbGluZSBnZW5lcmF0aW9uczoge2Jhc2VsaW5lX3RvdGFsOix9XG4iCiAgICAgICAgICAgIGYiICB0b3RhbDoge3N0ZWVyZWRfdG90YWwgKyBiYXNlbGluZV90b3RhbDosfSIKICAgICAgICApCg==',
    'data.py': '...',
    'evaluate.py': 'IiIiVGhlIHR3byBmdWxseS1hdXRvbWF0aWMgbWV0cmljczogQkVSVFNjb3JlIGFuZCBwZXJwbGV4aXR5LgoKICAgIGJlcnRzY29yZSAgIC0gaXMgdGhlIGFuc3dlciBzdGlsbCBhYm91dCB0aGUgcXVlc3Rpb24gdGhhdCB3YXMgYXNrZWQKICAgIHBlcnBsZXhpdHkgIC0gaXMgdGhlIGFuc3dlciBzdGlsbCBmbHVlbnQgRW5nbGlzaAoKVGhleSBtZWFzdXJlIGRpZmZlcmVudCB0aGluZ3Mgb24gcHVycG9zZS4gQW4gYW5zd2VyIGNhbiBiZSBmbHVlbnQgYnV0IG9mZgp0b3BpYywgb3Igb24gdG9waWMgYnV0IGdhcmJsZWQsIGFuZCBvbmUgYmxlbmRlZCAiY29oZXJlbmNlIiBudW1iZXIgd291bGQKaGlkZSBib3RoIGNhc2VzLgoKUGx1cmFsaXNtIGl0c2VsZiBpcyBub3Qgc2NvcmVkIGhlcmUsIHRoYXQgbmVlZHMgYSBqdWRnZS4gU2VlIGp1ZGdlLnB5IGZvcgp0aGUgcGFpcndpc2UgTExNLWFzLWp1ZGdlIHN5c3RlbS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCgpmcm9tIC4gaW1wb3J0IGNvbmZpZyBhcyBjZmcKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgQkVSVFNjb3JlIC0gcXVlc3Rpb24vYW5zd2VyIHJlbGV2YW5jZQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpfYmVydF9zY29yZXIgPSBOb25lCgoKZGVmIF9nZXRfYmVydF9zY29yZXIoKToKICAgIGdsb2JhbCBfYmVydF9zY29yZXIKICAgIGlmIF9iZXJ0X3Njb3JlciBpcyBOb25lOgogICAgICAgIGZyb20gYmVydF9zY29yZSBpbXBvcnQgQkVSVFNjb3JlcgoKICAgICAgICBkZXZpY2UgPSAiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiCiAgICAgICAgdHJ5OgogICAgICAgICAgICBfYmVydF9zY29yZXIgPSBCRVJUU2NvcmVyKAogICAgICAgICAgICAgICAgbW9kZWxfdHlwZT1jZmcuQkVSVFNDT1JFX01PREVMLAogICAgICAgICAgICAgICAgbGFuZz0iZW4iLAogICAgICAgICAgICAgICAgcmVzY2FsZV93aXRoX2Jhc2VsaW5lPVRydWUsCiAgICAgICAgICAgICAgICBkZXZpY2U9ZGV2aWNlLAogICAgICAgICAgICApCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgIHByaW50KGYiQkVSVFNjb3JlIGZhbGxpbmcgYmFjayB0byB7Y2ZnLkJFUlRTQ09SRV9GQUxMQkFDS30gKHtleGN9KSIpCiAgICAgICAgICAgIF9iZXJ0X3Njb3JlciA9IEJFUlRTY29yZXIoCiAgICAgICAgICAgICAgICBtb2RlbF90eXBlPWNmZy5CRVJUU0NPUkVfRkFMTEJBQ0ssCiAgICAgICAgICAgICAgICBsYW5nPSJlbiIsCiAgICAgICAgICAgICAgICByZXNjYWxlX3dpdGhfYmFzZWxpbmU9VHJ1ZSwKICAgICAgICAgICAgICAgIGRldmljZT1kZXZpY2UsCiAgICAgICAgICAgICkKCiAgICAgICAgIyBTb21lIHRva2VuaXplcnMgKERlQkVSVGEgaW5jbHVkZWQpIGRvbid0IHNldCBhIHJlYWwgbWF4IGxlbmd0aCwgc28KICAgICAgICAjIHRyYW5zZm9ybWVycyBmYWxscyBiYWNrIHRvIGEgc2VudGluZWwgbWVhbmluZyAidW5ib3VuZGVkIiwgYXJvdW5kCiAgICAgICAgIyAxMF4zMC4gYmVydF9zY29yZSBwYXNzZXMgdGhhdCBzdHJhaWdodCBpbnRvIHRoZSBmYXN0IHRva2VuaXplcidzCiAgICAgICAgIyB0cnVuY2F0aW9uIHNldHVwLCB3aGljaCBleHBlY3RzIGFuIG9yZGluYXJ5IGludGVnZXIgYW5kIG92ZXJmbG93cy4KICAgICAgICAjIENsYW1wIGl0IHRvIHRoZSBzdGFuZGFyZCBsZW5ndGggZm9yIHRoaXMgbW9kZWwgZmFtaWx5IGJlZm9yZSBhbnkKICAgICAgICAjIHNjb3JpbmcgaGFwcGVucy4KICAgICAgICB0b2tlbml6ZXIgPSBfYmVydF9zY29yZXIuX3Rva2VuaXplcgogICAgICAgIGlmIHRva2VuaXplci5tb2RlbF9tYXhfbGVuZ3RoID4gMTAwXzAwMDoKICAgICAgICAgICAgdG9rZW5pemVyLm1vZGVsX21heF9sZW5ndGggPSA1MTIKICAgIHJldHVybiBfYmVydF9zY29yZXIKCgpkZWYgYmVydHNjb3JlX3JlbGV2YW5jZShxdWVzdGlvbnMsIGFuc3dlcnMsIGJhdGNoX3NpemU6IGludCA9IDY0KSAtPiBucC5uZGFycmF5OgogICAgIiIiRjEgQkVSVFNjb3JlIG9mIGVhY2ggYW5zd2VyIGFnYWluc3QgaXRzIG93biBxdWVzdGlvbi4KCiAgICBIaWdoIG1lYW5zIHRoZSBhbnN3ZXIgaXMgc2VtYW50aWNhbGx5IGFib3V0IHRoZSBxdWVzdGlvbi4gVGhpcyBpcyBhCiAgICByZWxldmFuY2UgY2hlY2ssIG5vdCBhIHF1YWxpdHkgY2hlY2s6IGEgZmx1ZW50IGFuc3dlciB0byBhIGRpZmZlcmVudAogICAgcXVlc3Rpb24gc2NvcmVzIGxvdywgd2hpY2ggaXMgZXhhY3RseSB3aGF0IHNob3VsZCBiZSBjYXVnaHQgaGVyZS4KICAgICIiIgogICAgc2NvcmVyID0gX2dldF9iZXJ0X3Njb3JlcigpCgogICAgY2xlYW5fcXVlc3Rpb25zLCBjbGVhbl9hbnN3ZXJzLCBrZXB0X2luZGljZXMgPSBbXSwgW10sIFtdCiAgICBmb3IgaSwgKHF1ZXN0aW9uLCBhbnN3ZXIpIGluIGVudW1lcmF0ZSh6aXAocXVlc3Rpb25zLCBhbnN3ZXJzKSk6CiAgICAgICAgaWYgYW5zd2VyIGFuZCBzdHIoYW5zd2VyKS5zdHJpcCgpOgogICAgICAgICAgICBjbGVhbl9xdWVzdGlvbnMuYXBwZW5kKHN0cihxdWVzdGlvbikpCiAgICAgICAgICAgIGNsZWFuX2Fuc3dlcnMuYXBwZW5kKHN0cihhbnN3ZXIpKQogICAgICAgICAgICBrZXB0X2luZGljZXMuYXBwZW5kKGkpCgogICAgc2NvcmVzID0gbnAuZnVsbChsZW4oYW5zd2VycyksIG5wLm5hbiwgZHR5cGU9bnAuZmxvYXQzMikKICAgIGlmIG5vdCBjbGVhbl9hbnN3ZXJzOgogICAgICAgIHJldHVybiBzY29yZXMKCiAgICBfLCBfLCBmMSA9IHNjb3Jlci5zY29yZShjbGVhbl9hbnN3ZXJzLCBjbGVhbl9xdWVzdGlvbnMsIGJhdGNoX3NpemU9YmF0Y2hfc2l6ZSkKICAgIGZvciBpbmRleCwgdmFsdWUgaW4gemlwKGtlcHRfaW5kaWNlcywgZjEudG9saXN0KCkpOgogICAgICAgIHNjb3Jlc1tpbmRleF0gPSB2YWx1ZQogICAgcmV0dXJuIHNjb3JlcwoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBQZXJwbGV4aXR5IC0gZmx1ZW5jeQojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpfcGVycGxleGl0eV9tb2RlbCA9IE5vbmUKX3BlcnBsZXhpdHlfdG9rZW5pemVyID0gTm9uZQoKCmRlZiBfZ2V0X3BlcnBsZXhpdHlfc2NvcmVyKCk6CiAgICAiIiJPbmUgZml4ZWQgc2NvcmVyIGZvciBldmVyeSBtb2RlbCB1bmRlciB0ZXN0LgoKICAgIFNjb3JpbmcgYSBtb2RlbCdzIG93biBvdXRwdXQgd2l0aCBpdHNlbGYgbWFrZXMgdGhlIG51bWJlcnMgaW5jb21wYXJhYmxlCiAgICBhY3Jvc3MgbW9kZWxzOiBhIDNCIGFsd2F5cyBmaW5kcyBpdHMgb3duIHRleHQgbW9yZSBwcmVkaWN0YWJsZSB0aGFuIGEKICAgIDEzNU0gZmluZHMgaXRzIG93bi4gQSBzaW5nbGUgZXh0ZXJuYWwgc2NvcmVyIGtlZXBzIGV2ZXJ5dGhpbmcgb24gb25lIHNjYWxlLgogICAgIiIiCiAgICBnbG9iYWwgX3BlcnBsZXhpdHlfbW9kZWwsIF9wZXJwbGV4aXR5X3Rva2VuaXplcgogICAgaWYgX3BlcnBsZXhpdHlfbW9kZWwgaXMgTm9uZToKICAgICAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsRm9yQ2F1c2FsTE0sIEF1dG9Ub2tlbml6ZXIKCiAgICAgICAgZGV2aWNlID0gImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IgogICAgICAgIF9wZXJwbGV4aXR5X3Rva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKGNmZy5QRVJQTEVYSVRZX1NDT1JFUikKICAgICAgICBpZiBfcGVycGxleGl0eV90b2tlbml6ZXIucGFkX3Rva2VuIGlzIE5vbmU6CiAgICAgICAgICAgIF9wZXJwbGV4aXR5X3Rva2VuaXplci5wYWRfdG9rZW4gPSBfcGVycGxleGl0eV90b2tlbml6ZXIuZW9zX3Rva2VuCiAgICAgICAgX3BlcnBsZXhpdHlfbW9kZWwgPSBBdXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgICAgIGNmZy5QRVJQTEVYSVRZX1NDT1JFUiwKICAgICAgICAgICAgZHR5cGU9dG9yY2guZmxvYXQxNiBpZiBkZXZpY2UgPT0gImN1ZGEiIGVsc2UgdG9yY2guZmxvYXQzMiwKICAgICAgICApLnRvKGRldmljZSkKICAgICAgICBfcGVycGxleGl0eV9tb2RlbC5ldmFsKCkKICAgIHJldHVybiBfcGVycGxleGl0eV9tb2RlbCwgX3BlcnBsZXhpdHlfdG9rZW5pemVyCgoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgcGVycGxleGl0eSh0ZXh0cywgbWF4X2xlbmd0aDogaW50ID0gMjU2KSAtPiBucC5uZGFycmF5OgogICAgIiIiVG9rZW4tbGV2ZWwgcGVycGxleGl0eSB1bmRlciB0aGUgZml4ZWQgc2NvcmVyLiBMb3dlciBpcyBtb3JlIGZsdWVudC4KCiAgICBDb21wdXRlZCBvbmUgdGV4dCBhdCBhIHRpbWUgc28gcGFkZGluZyBuZXZlciBjb250cmlidXRlcyB0byB0aGUgbG9zcywKICAgIHdoaWNoIGlzIHRoZSB1c3VhbCBzb3VyY2Ugb2Ygd3JvbmcgcGVycGxleGl0eSBudW1iZXJzIGluIGJhdGNoZWQgY29kZS4KICAgICIiIgogICAgbW9kZWwsIHRva2VuaXplciA9IF9nZXRfcGVycGxleGl0eV9zY29yZXIoKQogICAgZGV2aWNlID0gbmV4dChtb2RlbC5wYXJhbWV0ZXJzKCkpLmRldmljZQoKICAgIHNjb3JlcyA9IG5wLmZ1bGwobGVuKHRleHRzKSwgbnAubmFuLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgZm9yIGksIHRleHQgaW4gZW51bWVyYXRlKHRleHRzKToKICAgICAgICBpZiBub3QgdGV4dCBvciBub3Qgc3RyKHRleHQpLnN0cmlwKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZW5jb2RlZCA9IHRva2VuaXplcigKICAgICAgICAgICAgc3RyKHRleHQpLCByZXR1cm5fdGVuc29ycz0icHQiLCB0cnVuY2F0aW9uPVRydWUsIG1heF9sZW5ndGg9bWF4X2xlbmd0aAogICAgICAgICkudG8oZGV2aWNlKQogICAgICAgIGlmIGVuY29kZWRbImlucHV0X2lkcyJdLnNoYXBlWzFdIDwgMjoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBsb3NzID0gbW9kZWwoKiplbmNvZGVkLCBsYWJlbHM9ZW5jb2RlZFsiaW5wdXRfaWRzIl0pLmxvc3MuaXRlbSgpCiAgICAgICAgaWYgbWF0aC5pc2Zpbml0ZShsb3NzKToKICAgICAgICAgICAgc2NvcmVzW2ldID0gZmxvYXQobWF0aC5leHAobWluKGxvc3MsIDIwLjApKSkKICAgIHJldHVybiBzY29yZXMKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgU2NvcmluZwojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpkZWYgc2NvcmVfZGF0YWZyYW1lKGdlbmVyYXRpb25zOiBwZC5EYXRhRnJhbWUpIC0+IHBkLkRhdGFGcmFtZToKICAgICIiIkF0dGFjaCBCRVJUU2NvcmUgYW5kIHBlcnBsZXhpdHkgdG8gYSBnZW5lcmF0aW9ucyB0YWJsZS4KCiAgICBFeHBlY3RzIGNvbHVtbnM6IHF1ZXN0aW9uLCByZXNwb25zZSwgcGx1cyB3aGF0ZXZlciBncm91cGluZyBjb2x1bW5zLgogICAgIiIiCiAgICBzY29yZWQgPSBnZW5lcmF0aW9ucy5jb3B5KCkKICAgIHNjb3JlZFsicmVzcG9uc2UiXSA9IHNjb3JlZFsicmVzcG9uc2UiXS5maWxsbmEoIiIpLmFzdHlwZShzdHIpCgogICAgcHJpbnQoZiJiZXJ0c2NvcmUgb24ge2xlbihzY29yZWQpOix9IHJvd3MiKQogICAgc2NvcmVkWyJiZXJ0c2NvcmUiXSA9IGJlcnRzY29yZV9yZWxldmFuY2Uoc2NvcmVkWyJxdWVzdGlvbiJdLnRvbGlzdCgpLCBzY29yZWRbInJlc3BvbnNlIl0udG9saXN0KCkpCgogICAgcHJpbnQoZiJwZXJwbGV4aXR5IG9uIHtsZW4oc2NvcmVkKTosfSByb3dzIikKICAgIHNjb3JlZFsicGVycGxleGl0eSJdID0gcGVycGxleGl0eShzY29yZWRbInJlc3BvbnNlIl0udG9saXN0KCkpCgogICAgc2NvcmVkWyJuX3dvcmRzIl0gPSBzY29yZWRbInJlc3BvbnNlIl0uc3RyLnNwbGl0KCkuc3RyLmxlbigpLmZpbGxuYSgwKS5hc3R5cGUoaW50KQogICAgcmV0dXJuIHNjb3JlZAoKCkdST1VQX0NPTFVNTlMgPSAoIm1vZGVsIiwgImF4aXMiLCAibWV0aG9kIiwgImFscGhhIiwgImNvbmRpdGlvbiIpCgoKZGVmIF9ncm91cF9jb2x1bW5zKGRmOiBwZC5EYXRhRnJhbWUpIC0+IGxpc3Q6CiAgICByZXR1cm4gW2MgZm9yIGMgaW4gR1JPVVBfQ09MVU1OUyBpZiBjIGluIGRmLmNvbHVtbnNdCgoKZGVmIHN1bW1hcmlzZShzY29yZWQ6IHBkLkRhdGFGcmFtZSkgLT4gcGQuRGF0YUZyYW1lOgogICAgIiIiQ29sbGFwc2Ugc2NvcmVkIGdlbmVyYXRpb25zIGludG8gdGhlIHRhYmxlIHRoYXQgZ29lcyBpbiB0aGUgcGFwZXIuCgogICAgQkVSVFNjb3JlIGlzIGF2ZXJhZ2VkLCBwZXJwbGV4aXR5IGlzIHRha2VuIGFzIGEgbWVkaWFuLiBUaGF0IGRpZmZlcmVuY2UKICAgIGlzIGRlbGliZXJhdGU6IGEgc2luZ2xlIGJyb2tlbiBnZW5lcmF0aW9uIGNhbiBzY29yZSBpbiB0aGUgdGhvdXNhbmRzIGFuZAogICAgd291bGQgZHJhZyBhIG1lYW4gcGVycGxleGl0eSB3aXRoIGl0LCB3aGlsZSBCRVJUU2NvcmUgaXMgYm91bmRlZCBhbmQKICAgIGF2ZXJhZ2VzIGNsZWFubHkuIFRoZSBgcGVycGxleGl0eWAgY29sdW1uIGJlbG93IGlzIHRoZXJlZm9yZSBhIG1lZGlhbiwKICAgIHVubGlrZSBldmVyeSBvdGhlciBhZ2dyZWdhdGUgaGVyZS4KICAgICIiIgogICAgZ3JvdXBfY29sdW1ucyA9IF9ncm91cF9jb2x1bW5zKHNjb3JlZCkgb3IgWyJjb25kaXRpb24iXQoKICAgIHN1bW1hcnkgPSBzY29yZWQuZ3JvdXBieShncm91cF9jb2x1bW5zLCBkcm9wbmE9RmFsc2UpLmFnZygKICAgICAgICBuPSgicmVzcG9uc2UiLCAic2l6ZSIpLAogICAgICAgIGJlcnRzY29yZT0oImJlcnRzY29yZSIsICJtZWFuIiksCiAgICAgICAgYmVydHNjb3JlX3NkPSgiYmVydHNjb3JlIiwgInN0ZCIpLAogICAgICAgIHBlcnBsZXhpdHk9KCJwZXJwbGV4aXR5IiwgIm1lZGlhbiIpLAogICAgICAgIG5fd29yZHM9KCJuX3dvcmRzIiwgIm1lYW4iKSwKICAgICAgICBlbXB0eV9yYXRlPSgicmVzcG9uc2UiLCBsYW1iZGEgczogZmxvYXQoKHMuc3RyLnN0cmlwKCkgPT0gIiIpLm1lYW4oKSkpLAogICAgKQogICAgcmV0dXJuIHN1bW1hcnkucmVzZXRfaW5kZXgoKS5zb3J0X3ZhbHVlcyhncm91cF9jb2x1bW5zKQo=',
    'judge.py': '...',
    'run_pipeline.py': '...',
    'steering.py': 'IiIiRm9yd2FyZC1ob29rIGFjdGl2YXRpb24gc3RlZXJpbmcgYW5kIGJhdGNoZWQgZ2VuZXJhdGlvbi4KClRoZSBob29rIGFkZHMgYWxwaGEgKiB2ZWN0b3IgdG8gdGhlIHJlc2lkdWFsIHN0cmVhbSBhdCBvbmUgbGF5ZXIuIFR3bwp0aGluZ3MgbWF0dGVyIGhlcmU6CgoxLiBUaGUgcHJvbXB0IGl0c2VsZiBpcyBsZWZ0IHVuc3RlZXJlZCwgZXhjZXB0IGl0cyBmaW5hbCBwb3NpdGlvbiwgd2hpY2gKICAgc2VlZHMgdGhlIGZpcnN0IGdlbmVyYXRlZCB0b2tlbi4gU3RlZXJpbmcgZXZlcnkgcHJvbXB0IHRva2VuIGRpc3RvcnRzCiAgIGhvdyB0aGUgbW9kZWwgcmVhZHMgdGhlIHF1ZXN0aW9uIGl0c2VsZiBhbmQgYnJlYWtzIGdyYW1tYXIgcmF0aGVyIHRoYW4KICAgc2hpZnRpbmcgdGhlIGFuc3dlcidzIGNvbnRlbnQuCgoyLiBUaGUgaG9vayBoYXMgdG8gaGFuZGxlIGJvdGggc2hhcGVzIGEgZGVjb2RlciBibG9jayBjYW4gcmV0dXJuOiBhIGJhcmUKICAgdGVuc29yLCBvciBhIHR1cGxlIHdob3NlIGZpcnN0IGVsZW1lbnQgaXMgdGhlIGhpZGRlbiBzdGF0ZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgY29udGV4dGxpYgoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAoKZnJvbSAuIGltcG9ydCBjb25maWcgYXMgY2ZnCmZyb20gLnZlY3RvcnMgaW1wb3J0IGxheWVyX21vZHVsZQoKCmNsYXNzIFN0ZWVyaW5nSG9vazoKICAgICIiIkFkZHMgYWxwaGEgKiB2ZWN0b3IgdG8gdGhlIHJlc2lkdWFsIHN0cmVhbSBhdCBvbmUgbGF5ZXIuCgogICAgRHVyaW5nIGdlbmVyYXRpb24sIHRoZSBmaXJzdCBmb3J3YXJkIHBhc3MgY292ZXJzIHRoZSB3aG9sZSBwcm9tcHQgYW5kCiAgICBldmVyeSBwYXNzIGFmdGVyIHRoYXQgY292ZXJzIGEgc2luZ2xlIG5ldyB0b2tlbi4gVGhpcyB0cmFja3Mgd2hpY2gKICAgIHBhc3MgaXQncyBvbiBzbyB0aGUgcHJvbXB0IGlzIGxlZnQgdW50b3VjaGVkIHdoZW4gc3RlZXJfcHJvbXB0IGlzIEZhbHNlLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHZlY3RvcjogdG9yY2guVGVuc29yLCBhbHBoYTogZmxvYXQsIHN0ZWVyX3Byb21wdDogYm9vbCA9IEZhbHNlKToKICAgICAgICBzZWxmLnZlY3RvciA9IHZlY3RvcgogICAgICAgIHNlbGYuYWxwaGEgPSBhbHBoYQogICAgICAgIHNlbGYuc3RlZXJfcHJvbXB0ID0gc3RlZXJfcHJvbXB0CiAgICAgICAgc2VsZi5pc19maXJzdF9wYXNzID0gVHJ1ZQoKICAgIGRlZiByZXNldChzZWxmKToKICAgICAgICBzZWxmLmlzX2ZpcnN0X3Bhc3MgPSBUcnVlCgogICAgZGVmIF9fY2FsbF9fKHNlbGYsIG1vZHVsZSwgaW5wdXRzLCBvdXRwdXQpOgogICAgICAgIHJldHVybnNfdHVwbGUgPSBpc2luc3RhbmNlKG91dHB1dCwgdHVwbGUpCiAgICAgICAgaGlkZGVuID0gb3V0cHV0WzBdIGlmIHJldHVybnNfdHVwbGUgZWxzZSBvdXRwdXQKCiAgICAgICAgaWYgc2VsZi5hbHBoYSA9PSAwLjA6CiAgICAgICAgICAgIHNlbGYuaXNfZmlyc3RfcGFzcyA9IEZhbHNlCiAgICAgICAgICAgIHJldHVybiBvdXRwdXQKCiAgICAgICAgZGVsdGEgPSAoc2VsZi5hbHBoYSAqIHNlbGYudmVjdG9yKS50byhkdHlwZT1oaWRkZW4uZHR5cGUsIGRldmljZT1oaWRkZW4uZGV2aWNlKQoKICAgICAgICBpZiBzZWxmLmlzX2ZpcnN0X3Bhc3MgYW5kIG5vdCBzZWxmLnN0ZWVyX3Byb21wdDoKICAgICAgICAgICAgIyBQcm9tcHQgcGFzczogc3RlZXIgb25seSB0aGUgbGFzdCBwb3NpdGlvbiwgc2luY2UgdGhhdCdzIHRoZQogICAgICAgICAgICAjIG9uZSB0aGF0IHNlZWRzIHRoZSBmaXJzdCBnZW5lcmF0ZWQgdG9rZW4uCiAgICAgICAgICAgIGhpZGRlbiA9IGhpZGRlbi5jbG9uZSgpCiAgICAgICAgICAgIGhpZGRlbls6LCAtMSwgOl0gPSBoaWRkZW5bOiwgLTEsIDpdICsgZGVsdGEKICAgICAgICBlbHNlOgogICAgICAgICAgICBoaWRkZW4gPSBoaWRkZW4gKyBkZWx0YQoKICAgICAgICBzZWxmLmlzX2ZpcnN0X3Bhc3MgPSBGYWxzZQogICAgICAgIHJldHVybiAoaGlkZGVuLCkgKyBvdXRwdXRbMTpdIGlmIHJldHVybnNfdHVwbGUgZWxzZSBoaWRkZW4KCgpAY29udGV4dGxpYi5jb250ZXh0bWFuYWdlcgpkZWYgc3RlZXJpbmcobW9kZWwsIGxheWVyX2lkeDogaW50LCB2ZWN0b3IsIGFscGhhOiBmbG9hdCwgc3RlZXJfcHJvbXB0OiBib29sIHwgTm9uZSA9IE5vbmUpOgogICAgIiIiQXR0YWNoIGEgc3RlZXJpbmcgaG9vayBmb3IgdGhlIGR1cmF0aW9uIG9mIHRoZSBibG9jay4iIiIKICAgIGlmIHN0ZWVyX3Byb21wdCBpcyBOb25lOgogICAgICAgIHN0ZWVyX3Byb21wdCA9IGNmZy5TVEVFUl9QUk9NUFRfVE9LRU5TCgogICAgaWYgaXNpbnN0YW5jZSh2ZWN0b3IsIG5wLm5kYXJyYXkpOgogICAgICAgIHZlY3RvciA9IHRvcmNoLmZyb21fbnVtcHkodmVjdG9yKQogICAgdmVjdG9yID0gdmVjdG9yLmZsb2F0KCkKCiAgICBob29rID0gU3RlZXJpbmdIb29rKHZlY3RvciwgYWxwaGEsIHN0ZWVyX3Byb21wdCkKICAgIGhhbmRsZSA9IGxheWVyX21vZHVsZShtb2RlbCwgbGF5ZXJfaWR4KS5yZWdpc3Rlcl9mb3J3YXJkX2hvb2soaG9vaykKICAgIHRyeToKICAgICAgICB5aWVsZCBob29rCiAgICBmaW5hbGx5OgogICAgICAgIGhhbmRsZS5yZW1vdmUoKQoKCmRlZiBidWlsZF9jaGF0X3Byb21wdHModG9rZW5pemVyLCBxdWVzdGlvbnMsIHN5c3RlbTogc3RyIHwgTm9uZSA9IE5vbmUpOgogICAgIiIiQXBwbHkgdGhlIG1vZGVsJ3MgY2hhdCB0ZW1wbGF0ZSBzbyBpbnN0cnVjdCBtb2RlbHMgYmVoYXZlIHByb3Blcmx5LiIiIgogICAgcHJvbXB0cyA9IFtdCiAgICBmb3IgcXVlc3Rpb24gaW4gcXVlc3Rpb25zOgogICAgICAgIG1lc3NhZ2VzID0gKFt7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBzeXN0ZW19XSBpZiBzeXN0ZW0gZWxzZSBbXSkgKyBbCiAgICAgICAgICAgIHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiBxdWVzdGlvbn0KICAgICAgICBdCiAgICAgICAgdHJ5OgogICAgICAgICAgICBwcm9tcHRfdGV4dCA9IHRva2VuaXplci5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgICAgICAgICAgbWVzc2FnZXMsIHRva2VuaXplPUZhbHNlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZQogICAgICAgICAgICApCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcHJvbXB0X3RleHQgPSAoZiJ7c3lzdGVtfVxuXG4iIGlmIHN5c3RlbSBlbHNlICIiKSArIGYiVXNlcjoge3F1ZXN0aW9ufVxuQXNzaXN0YW50OiIKICAgICAgICBwcm9tcHRzLmFwcGVuZChwcm9tcHRfdGV4dCkKICAgIHJldHVybiBwcm9tcHRzCgoKQHRvcmNoLm5vX2dyYWQoKQpkZWYgZ2VuZXJhdGUoCiAgICBtb2RlbCwKICAgIHRva2VuaXplciwKICAgIHF1ZXN0aW9ucywKICAgIGRldmljZTogc3RyLAogICAgbGF5ZXJfaWR4OiBpbnQgfCBOb25lID0gTm9uZSwKICAgIHZlY3Rvcj1Ob25lLAogICAgYWxwaGE6IGZsb2F0ID0gMC4wLAogICAgYmF0Y2hfc2l6ZTogaW50IHwgTm9uZSA9IE5vbmUsCiAgICBtYXhfbmV3X3Rva2VuczogaW50IHwgTm9uZSA9IE5vbmUsCiAgICBzZWVkOiBpbnQgfCBOb25lID0gTm9uZSwKKToKICAgICIiIkdlbmVyYXRlIGFuc3dlcnMsIG9wdGlvbmFsbHkgc3RlZXJlZC4gUmV0dXJucyBhIGxpc3Qgb2Ygc3RyaW5ncy4KCiAgICBQYXNzaW5nIHZlY3Rvcj1Ob25lIG9yIGFscGhhPTAgZ2l2ZXMgdGhlIHVuc3RlZXJlZCBiYXNlbGluZS4KICAgICIiIgogICAgYmF0Y2hfc2l6ZSA9IGJhdGNoX3NpemUgb3IgY2ZnLkdFTkVSQVRJT05fQkFUQ0hfU0laRQogICAgbWF4X25ld190b2tlbnMgPSBtYXhfbmV3X3Rva2VucyBvciBjZmcuTUFYX05FV19UT0tFTlMKICAgIHNlZWQgPSBjZmcuU0VFRCBpZiBzZWVkIGlzIE5vbmUgZWxzZSBzZWVkCgogICAgcHJvbXB0cyA9IGJ1aWxkX2NoYXRfcHJvbXB0cyh0b2tlbml6ZXIsIHF1ZXN0aW9ucykKICAgIGFuc3dlcnMgPSBbXQoKICAgIHNob3VsZF9zdGVlciA9IHZlY3RvciBpcyBub3QgTm9uZSBhbmQgYWxwaGEgIT0gMC4wCiAgICBzdGVlcmluZ19jb250ZXh0ID0gKAogICAgICAgIHN0ZWVyaW5nKG1vZGVsLCBsYXllcl9pZHgsIHZlY3RvciwgYWxwaGEpCiAgICAgICAgaWYgc2hvdWxkX3N0ZWVyCiAgICAgICAgZWxzZSBjb250ZXh0bGliLm51bGxjb250ZXh0KE5vbmUpCiAgICApCgogICAgd2l0aCBzdGVlcmluZ19jb250ZXh0IGFzIGhvb2s6CiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIGxlbihwcm9tcHRzKSwgYmF0Y2hfc2l6ZSk6CiAgICAgICAgICAgIGJhdGNoID0gcHJvbXB0c1tzdGFydCA6IHN0YXJ0ICsgYmF0Y2hfc2l6ZV0KICAgICAgICAgICAgZW5jb2RlZCA9IHRva2VuaXplcigKICAgICAgICAgICAgICAgIGJhdGNoLCByZXR1cm5fdGVuc29ycz0icHQiLCBwYWRkaW5nPVRydWUsIHRydW5jYXRpb249VHJ1ZSwgbWF4X2xlbmd0aD01MTIKICAgICAgICAgICAgKS50byhkZXZpY2UpCgogICAgICAgICAgICBpZiBob29rIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgaG9vay5yZXNldCgpCgogICAgICAgICAgICB0b3JjaC5tYW51YWxfc2VlZChzZWVkICsgc3RhcnQpCiAgICAgICAgICAgIGdlbmVyYXRlZCA9IG1vZGVsLmdlbmVyYXRlKAogICAgICAgICAgICAgICAgKiplbmNvZGVkLAogICAgICAgICAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgICAgICAgICBkb19zYW1wbGU9Y2ZnLlRFTVBFUkFUVVJFID4gMCwKICAgICAgICAgICAgICAgIHRlbXBlcmF0dXJlPWNmZy5URU1QRVJBVFVSRSwKICAgICAgICAgICAgICAgIHRvcF9wPWNmZy5UT1BfUCwKICAgICAgICAgICAgICAgIHBhZF90b2tlbl9pZD10b2tlbml6ZXIucGFkX3Rva2VuX2lkLAogICAgICAgICAgICApCgogICAgICAgICAgICAjIFN0cmlwIHRoZSBwcm9tcHQgYmFjayBvZmY7IGxlZnQgcGFkZGluZyBtYWtlcyBpdCBhIGZpeGVkLXdpZHRoIHByZWZpeC4KICAgICAgICAgICAgbmV3X3Rva2VucyA9IGdlbmVyYXRlZFs6LCBlbmNvZGVkWyJpbnB1dF9pZHMiXS5zaGFwZVsxXSA6XQogICAgICAgICAgICBkZWNvZGVkID0gdG9rZW5pemVyLmJhdGNoX2RlY29kZShuZXdfdG9rZW5zLCBza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUpCiAgICAgICAgICAgIGFuc3dlcnMuZXh0ZW5kKHRleHQuc3RyaXAoKSBmb3IgdGV4dCBpbiBkZWNvZGVkKQoKICAgIHJldHVybiBhbnN3ZXJzCg==',
    'vectors.py': '...',
}

def _find_existing():
    cands = ['/content/drive/MyDrive/MIORPA PROJECT',
             '/content/MIORPA PROJECT', os.getcwd()]
    cands += [os.path.dirname(os.path.dirname(p))
              for p in glob.glob('/content/**/miorpa/config.py', recursive=True)]
    for c in cands:
        if c and os.path.isfile(os.path.join(c, 'miorpa', 'config.py')):
            return c
    return None

# Mount Drive and use it as the working directory. Everything the pipeline
# writes - dataset, vectors, generations, evaluation CSVs - then lands in
# Drive as it's produced, so it survives a disconnect and syncs to your
# computer through Drive itself instead of sitting on the throwaway runtime
# disk waiting to be manually downloaded before the session ends.
IN_COLAB = True
try:
    import google.colab  # noqa: F401
    if not os.path.isdir('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False

# Drive wins whenever it is available, ahead of anything _find_existing
# turns up. Its candidate list includes the working directory and a glob
# under /content, so a stray miorpa folder on the temporary disk could
# otherwise capture the run and send every result somewhere that does not
# survive the session.
if IN_COLAB and os.path.isdir('/content/drive/MyDrive'):
    PROJECT_DIR = '/content/drive/MyDrive/MIORPA PROJECT'
else:
    PROJECT_DIR = _find_existing()
    if PROJECT_DIR is None:
        PROJECT_DIR = '/content' if os.path.isdir('/content') else os.getcwd()

# The embedded copy always wins. An earlier version preferred whatever
# package was already on disk, to protect local edits - but once the
# project lives in Drive that folder survives between sessions, so every
# run silently executed stale code while the notebook looked current.
# That cost a full judging run. Now anything that differs is overwritten
# and named, so a mismatch is visible instead of silent.
target = os.path.join(PROJECT_DIR, 'miorpa')
os.makedirs(target, exist_ok=True)
written, refreshed = [], []
for name, blob in _EMBEDDED.items():
    payload = base64.b64decode(blob)
    path = os.path.join(target, name)
    if os.path.isfile(path):
        with open(path, 'rb') as fh:
            if fh.read() == payload:
                continue
        refreshed.append(name)
    else:
        written.append(name)
    with open(path, 'wb') as fh:
        fh.write(payload)

# Three distinct outcomes, each said plainly. Collapsing 'wrote it fresh'
# into 'already matches' would be the same class of misleading
# confirmation that let stale code run unnoticed in the first place.
if written:
    print(f'wrote {len(written)} modules to {target}')
if refreshed:
    print('REFRESHED stale modules: ' + ', '.join(sorted(refreshed)))
    print('the copy on disk was older than this notebook and has been replaced')
if not written and not refreshed:
    print(f'package in {target} already matches the notebook')

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Drop any half-imported copy so a re-run picks up the files just written.
for mod in [m for m in list(sys.modules) if m == 'miorpa' or m.startswith('miorpa.')]:
    del sys.modules[mod]

print('working in', PROJECT_DIR)
print('package present:', os.path.isfile('miorpa/config.py'))
if PROJECT_DIR.startswith('/content/drive/'):
    print('results are saving to Google Drive - safe if this session drops')
elif IN_COLAB:
    print('WARNING: results are saving to the temporary Colab disk, not Drive.')
    print('They will be lost if this session disconnects. Download them with')
    print('the export cell before ending the session.')

have = {f: os.path.isfile(f'dataset/{f}.jsonl')
        for f in ('survey', 'conversations', 'utterances')}
print('dataset present:', all(have.values()), have)
if not all(have.values()):
    print('PRISM will be downloaded from Hugging Face in Step 1b')

In [ ]:
from miorpa import config as cfg
from miorpa import run_pipeline as rp
from miorpa import data, vectors, steering, evaluate, benchmarks, judge

import importlib
for module in (cfg, data, vectors, steering, evaluate, benchmarks, judge, rp):
    importlib.reload(module)

print('models available:')
for model in cfg.MODELS:
    print(f'  {model.short_name:16s} {model.params:>5s}  {model.hf_id}')

print(f'\ngeneration batch size, picked from the attached GPU: {cfg.GENERATION_BATCH_SIZE}')

## Step 1b: dataset

Downloads PRISM (about 128 MB) from Hugging Face if it isn't already on disk.

If it's gated for your account: accept the terms on the dataset page, then uncomment and run the login line below.

In [ ]:
# if PRISM is gated for your account, uncomment and run this once:
# from huggingface_hub import notebook_login; notebook_login()

dataset_ready = data.ensure_dataset()
print('\ndataset ready:', dataset_ready)

## Step 2: calibration pairs

For each axis (origin, religion, age) this finds pairs of answers to a similar question: one both sides rated highly (balanced), one only one side rated highly (one-sided). That pair is what the steering vector gets built from later.

Questions are matched by exact text first, then by meaning (cosine similarity 0.75 or above) for whatever's left. That second pass is what makes a thin axis like age usable at all.

Two sanity checks run after: is one side's answers systematically longer than the other (a possible confound), and does the split just track which AI model wrote the answer instead of which group liked it.

In [ ]:
pairs = rp.stage_pairs()

for axis, pairs_for_axis in pairs.items():
    if len(pairs_for_axis):
        print(f"\n{axis}: {len(pairs_for_axis):,} pairs")
        print(pairs_for_axis['match_type'].value_counts().to_string())

print('\n' + '=' * 60)
print('CONFOUND CHECK')
print('=' * 60)
for axis, pairs_for_axis in pairs.items():
    data.diagnose_pairs(pairs_for_axis, axis)

print('\n' + '=' * 60)
print('MODEL CLUSTERING')
print('=' * 60)
for axis, pairs_for_axis in pairs.items():
    data.diagnose_model_clustering(pairs_for_axis, axis)

# if any axis looks thin, this probably loaded an old cached file.
# force a rebuild with:
#   pairs = rp.stage_pairs(force=True)

## Step 3: test questions

Builds the questions used later to check whether steering worked. Pulled from the World Values Survey and Anthropic's persona evaluations.

Any test question too close to a calibration question from Step 2 (cosine 0.50 or above) gets removed. Otherwise the model could just be repeating what it saw during calibration instead of generalising to something new.

In [ ]:
eval_sets = rp.stage_benchmarks(per_axis=cfg.EVAL_QUESTIONS_PER_AXIS)

for axis, questions in eval_sets.items():
    print(f"\n{axis} ({len(questions)} questions), first 3:")
    for question in questions[:3]:
        print('  -', question)

## Step 4: steering vectors

This builds the actual steering vector: the direction inside the model's activations that stands for "pluralism" on a given axis. Four ways to compute it, per model per axis (full detail in miorpa/README.md):

- MoD, mean difference between balanced and one-sided activations. Simplest, and the one the reference paper (Im and Li, 2026) found works best.
- PoD and PoE, two PCA-based variants.
- CoE, a linear classifier boundary between the two sets.

Plus a random vector of the same size, as a control. If a random direction moves the output as much as a real one, the effect isn't real.

Injected 45% of the way through the network, based on the reference paper's own tuned result. Step 9's layer sweep checks whether that holds for our models too.

Runs across all six models listed in the next cell, one at a time.

In [ ]:
# All six small models, smallest first. Each one runs completely - vectors,
# generation, evaluation, then both judges - before the next one starts, so
# a disconnect only costs whichever model was in progress, not the set.
MODELS_TO_RUN = [
    'SmolLM2-135M',
    'Qwen2.5-0.5B',
    'SmolLM2-360M',
    'Qwen2.5-1.5B',
    'SmolLM2-1.7B',
    'Qwen2.5-3B',
]
print('running, in order:', MODELS_TO_RUN)

## Step 5: smoke test

Runs one question through the model twice, once plain and once steered. Read both answers before running the full batch. If something's broken here, it's broken 30,000 times over downstream.

In [ ]:
spec = cfg.MODELS_BY_NAME[MODELS_TO_RUN[0]]
smoke_bundle = rp.stage_vectors(pairs, model_names=[spec.short_name])[spec.short_name]
axis = 'origin'

model, tokenizer, device = vectors.load_model(spec)
layer = smoke_bundle['layer']
steering_vector = vectors.get_vector(smoke_bundle, axis, 'MoD')

test_question = ['Should economic growth be prioritised over environmental protection?']

baseline_answer = steering.generate(model, tokenizer, test_question, device, layer_idx=layer, vector=None, alpha=0.0)
steered_answer = steering.generate(model, tokenizer, test_question, device, layer_idx=layer, vector=steering_vector, alpha=cfg.DEFAULT_ALPHA)

print('QUESTION:', test_question[0])
print('\n--- BASELINE ---')
print(baseline_answer[0])
print(f'\n--- STEERED (MoD, alpha={cfg.DEFAULT_ALPHA}, layer {layer}) ---')
print(steered_answer[0])

vectors.free(model)

## Step 6-7-11: generation, evaluation, and judging - per model

Runs every model in `MODELS_TO_RUN` completely, one at a time: builds its steering vectors, generates every condition, scores with BERTScore/perplexity, then judges every pair with both Mistral and Prometheus. Results save to Drive as each stage finishes.

Safe to stop and restart: generation checkpoints per condition, judging checkpoints per batch. Re-running skips whatever is already done, model by model - a finished model is never redone just because a later one hasn't started yet.

In [ ]:
import pandas as pd

all_scored, all_summary, all_pairs = [], [], []
all_resolved_mistral, all_resolved_prometheus = [], []

for model_name in MODELS_TO_RUN:
    print('\n' + '#' * 70)
    print(f'# {model_name}')
    print('#' * 70)

    spec = cfg.MODELS_BY_NAME[model_name]
    bundle = rp.stage_vectors(pairs, model_names=[model_name])

    run_cfg = cfg.RunConfig(
        models=[model_name],
        axes=cfg.AXES,
        methods=('MoD', 'PoD', 'PoE', 'CoE'),
        alphas=(cfg.DEFAULT_ALPHA,),
        questions_per_axis=cfg.EVAL_QUESTIONS_PER_AXIS,
        include_random_control=True,
        tag=f'main_{spec.slug}',
    )
    print(run_cfg.describe())

    rp.stage_generate(run_cfg, eval_sets, bundle)
    scored_m, summary_m = rp.stage_evaluate(run_cfg)
    all_scored.append(scored_m)
    all_summary.append(summary_m)

    pairs_m = judge.build_pairwise_set(scored_m, tag=run_cfg.tag)
    # pair_id resets to P000000 for every model's own build_pairwise_set
    # call, so without this prefix two different models could share the
    # same pair_id and get silently merged together once everything below
    # is concatenated into one combined table.
    pairs_m['pair_id'] = spec.slug + '_' + pairs_m['pair_id'].astype(str)
    all_pairs.append(pairs_m)

    judged_mistral_m = judge.run_judge(pairs_m, "mistral", tag=run_cfg.tag)
    resolved_mistral_m = judge.resolve_pairwise(judged_mistral_m, tag=run_cfg.tag)
    judge.unload_judge()

    judged_prometheus_m = judge.run_judge(pairs_m, "prometheus", tag=run_cfg.tag)
    resolved_prometheus_m = judge.resolve_pairwise(judged_prometheus_m, tag=run_cfg.tag)
    judge.unload_judge()

    all_resolved_mistral.append(resolved_mistral_m)
    all_resolved_prometheus.append(resolved_prometheus_m)

    print(f'\n--- {model_name} win rates (mistral) ---')
    display(judge.summarise_pairwise(resolved_mistral_m))
    print(f'\n=== {model_name} finished, results saved ===\n')

scored = pd.concat(all_scored, ignore_index=True)
summary = pd.concat(all_summary, ignore_index=True)
pairs_judged = pd.concat(all_pairs, ignore_index=True)
resolved_mistral = pd.concat(all_resolved_mistral, ignore_index=True)
resolved_prometheus = pd.concat(all_resolved_prometheus, ignore_index=True)

scored.to_csv(cfg.EVAL_DIR / 'scored_main.csv', index=False, encoding='utf-8')
summary.to_csv(cfg.EVAL_DIR / 'summary_main.csv', index=False, encoding='utf-8')

run = cfg.RunConfig(models=MODELS_TO_RUN, tag='main')
print('\nall models finished. combined tables: scored, summary, resolved_mistral, resolved_prometheus')

In [ ]:
# generation now happens inside the per-model loop above.
print('generation completed above, per model')

## Step 7: evaluation - already run above

Scored inside the per-model loop above, right after generation. `summary` below is the combined table across all six models. Two automatic metrics:

| metric | checks | good direction |
|---|---|---|
| bertscore | did the answer stay on topic | higher |
| perplexity | is the answer still fluent English | lower |

Pluralism itself isn't scored here, that needs a judge - also already run above, in the same loop.

In [ ]:
# evaluation now happens inside the per-model loop above; scored and
# summary here are already the combined tables across every model.
summary

## Step 8: read the result

This only shows whether steering broke anything, not whether it worked. If bertscore drops or perplexity climbs sharply for a condition, that model/axis/method combination is degrading, and no judge score later will fix that.

Whether pluralism actually improved needs the judge results from Step 11.

In [ ]:
import pandas as pd

pivot = summary.pivot_table(
    index=['model', 'axis'],
    columns='method',
    values=['bertscore', 'perplexity', 'n_words'],
)
display(pivot.round(3))

baseline_rows = summary[summary['method'] == 'baseline'].set_index(['model', 'axis'])
steered_rows = summary[summary['method'] != 'baseline']

deltas_list = []
for _, row in steered_rows.iterrows():
    key = (row['model'], row['axis'])
    if key not in baseline_rows.index:
        continue
    baseline = baseline_rows.loc[key]
    deltas_list.append({
        'model': row['model'], 'axis': row['axis'], 'method': row['method'],
        'bertscore_change': row['bertscore'] - baseline['bertscore'],
        'perplexity_ratio': row['perplexity'] / baseline['perplexity'] if baseline['perplexity'] else float('nan'),
        'empty_rate': row['empty_rate'],
    })

deltas = pd.DataFrame(deltas_list).sort_values('bertscore_change', ascending=False)
print('\nchange against each model/axis baseline:')
display(deltas.round(3))

degraded = deltas[(deltas['bertscore_change'] < -0.05)
                   | (deltas['perplexity_ratio'] > 1.5)
                   | (deltas['empty_rate'] > 0.05)]
if len(degraded):
    print('\nsteering degraded the output here, lower alpha for these conditions:')
    display(degraded.round(3))
else:
    print('\nno condition degraded bertscore or perplexity noticeably')

if 'Random' in set(summary['method']):
    print('\nrandom control vs MoD (automatic metrics only):')
    display(summary[summary['method'].isin(['baseline', 'MoD', 'Random'])]
            [['model', 'axis', 'method', 'bertscore', 'perplexity', 'n_words']].round(3))

## Step 8b: is the axis gap real, or just a scaling artefact?

The main run gave every axis the same alpha. But alpha only multiplies the vector, and the vectors are not the same length: on Qwen2.5-1.5B, |MoD| is 14 for origin, 40 for age and 85 for religion. What the model actually receives is alpha times that length, so religion was pushed about six times harder than origin. The axes were never compared at the same strength.

Two cells here. The first measures how big that push is next to the hidden state it is added to, which is what says whether the model was nudged or drowned. The second runs the naive setup and the corrected one back to back, so the difference between them is measured rather than assumed.

In [ ]:
import pandas as pd

# |alpha * v| means nothing without something to compare it against. This
# reports it as a percentage of the typical |h| at the injection layer.
ratio_frames = []
for model_name in MODELS_TO_RUN:
    print(f'\n--- push ratio: {model_name} ---')
    ratio_frames.append(rp.measure_push_ratio(model_name, pairs).assign(model=model_name))

push_ratio = pd.concat(ratio_frames, ignore_index=True)
display(push_ratio)

In [ ]:
import pandas as pd

# Same model, same questions, two conditions: one shared alpha for every
# axis, then alpha rescaled per axis so the push is equal everywhere.
comparison_frames = []
for model_name in MODELS_TO_RUN:
    print(f'\n--- push comparison: {model_name} ---')
    comparison_frames.append(rp.ablation_equalised_push(model_name).assign(model_run=model_name))

push_comparison = pd.concat(comparison_frames, ignore_index=True)

steered = push_comparison[push_comparison['method'] == 'MoD']
display(steered.pivot_table(index=['model_run', 'axis'], columns='push_mode',
                            values=['bertscore', 'perplexity']).round(3))

print('\nIf religion and age recover once the push is equalised, the gap in the')
print('main run was a scaling artefact. If they stay flat at a strength that')
print('leaves origin coherent, the axes genuinely differ and that is the finding.')

## Step 9: ablations

Three checks, run on every model in `MODELS_TO_RUN`, one at a time:

- Random control: does a random vector move the output as much as the real one? If yes, the effect isn't real.
- Alpha sweep: how strong can steering get before the model breaks?
- Layer sweep: which layer actually carries the cultural signal, and does that answer change with model size?

Without a judge these only show whether the vector perturbs the model differently from noise, not whether it perturbs it toward pluralism. That needs the judge scores from the step above. Cheap relative to the main run - roughly 15 minutes per model for all three checks combined - so running the full set is worth it rather than guessing from one model.

In [ ]:
import pandas as pd

# ablations now run on every model in MODELS_TO_RUN, not just one - cheap
# enough relative to the main run that there's no reason to guess from a
# single model's curve.
all_control_scored, all_control_summary = [], []
for model_name in MODELS_TO_RUN:
    print(f'\n--- random control: {model_name} ---')
    control_scored_m, control_summary_m = rp.ablation_random_control(model_name)
    all_control_scored.append(control_scored_m)
    all_control_summary.append(control_summary_m)

control_scored = pd.concat(all_control_scored, ignore_index=True)
control_summary = pd.concat(all_control_summary, ignore_index=True)

In [ ]:
import pandas as pd

all_alpha_scored, all_alpha_summary = [], []
for model_name in MODELS_TO_RUN:
    print(f'\n--- alpha sweep: {model_name} ---')
    alpha_scored_m, alpha_summary_m = rp.ablation_alpha_sweep(model_name)
    all_alpha_scored.append(alpha_scored_m)
    all_alpha_summary.append(alpha_summary_m)

alpha_scored = pd.concat(all_alpha_scored, ignore_index=True)
alpha_summary = pd.concat(all_alpha_summary, ignore_index=True)

curve = alpha_summary.groupby(['model', 'alpha'])[['bertscore', 'perplexity', 'n_words', 'empty_rate']].mean()
display(curve.round(3))

print('\nThis curve gives the upper bound on alpha, not the optimum, per model.')
print('Where perplexity starts climbing and bertscore starts falling is where')
print('that model is breaking. The best alpha below that line is a pluralism')
print('question, which needs the judge scores from the step above.')

In [ ]:
import pandas as pd

all_layer_summary = []
for model_name in MODELS_TO_RUN:
    print(f'\n--- layer sweep: {model_name} ---')
    layer_summary_m = rp.ablation_layer_sweep(model_name)
    all_layer_summary.append(layer_summary_m)

layer_summary = pd.concat(all_layer_summary, ignore_index=True)
display(layer_summary.groupby(['model', 'layer_fraction'])[['bertscore', 'perplexity', 'n_words']].mean().round(3))

print('\nIm and Li steer Llama-2-7b-chat at layer 13 of 32, about 0.41 depth.')
print('config.LAYER_DEPTH_FRACTION is set to 0.45 on that basis. This sweep')
print('checks whether that holds across model sizes here too, not just one.')

## Step 10: export

Saves everything to `MIORPA_results.xlsx` - generations, scores, and judge win rates across all six models - for the paper and the shared sheet.

In [ ]:
import pandas as pd

output_path = cfg.RESULTS_DIR / 'MIORPA_results.xlsx'

winrates_mistral = judge.summarise_pairwise(resolved_mistral)
winrates_prometheus = judge.summarise_pairwise(resolved_prometheus)

optional_sheets = {
    'random_control': globals().get('control_summary'),
    'alpha_sweep': globals().get('alpha_summary'),
    'layer_sweep': globals().get('layer_summary'),
    'push_ratio': globals().get('push_ratio'),
    'push_comparison': globals().get('push_comparison'),
}

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    summary.to_excel(writer, sheet_name='main_summary', index=False)
    deltas.to_excel(writer, sheet_name='deltas_vs_baseline', index=False)
    scored.head(3000).to_excel(writer, sheet_name='generations_sample', index=False)
    winrates_mistral.to_excel(writer, sheet_name='judge_winrates_mistral', index=False)
    winrates_prometheus.to_excel(writer, sheet_name='judge_winrates_prometheus', index=False)
    for sheet_name, sheet_data in optional_sheets.items():
        if sheet_data is not None:
            sheet_data.to_excel(writer, sheet_name=sheet_name, index=False)
        else:
            print(f'{sheet_name}: not run yet, skipped')

print('saved', output_path)

## Step 11: pairwise LLM judging - already run above

Judging happened inside the per-model loop above, one model at a time, so a disconnect only costs the model in progress rather than the whole judged set. The cells below just show the combined results across every model that has finished so far.

This is the part the automatic metrics can't do: is the steered answer actually more pluralistic than the baseline, not just still coherent.

Two judges, neither from a model family under test (three of the six models are Qwen, so a Qwen judge would rate its own family higher and quietly bias the comparison):

- **Mistral-Small-24B-Instruct**, ungated, no login needed
- **Prometheus 2** (7B), built specifically for rubric-based scoring

Judging is pairwise, not an absolute 1-10 score: for the same question, the judge is shown the baseline answer and a steered answer, and picks which one better presents multiple cultural viewpoints, A, B, or tie. Every pair is judged in both orders, so a judge that just follows position rather than content shows up as picking the same letter both times, which gets flagged automatically.

Aya Expanse 32B is also registered (key `"aya"`) as a stronger, multilingual-native alternative, but it's gated and needs a Hugging Face login, so it isn't the default here, swap it in once that access is set up.

In [ ]:
# pairs were built and judged per model inside the loop above.
# pairs_judged is the combined set across every model that's finished so far.
pairs_judged.head()

In [ ]:
# mistral judging already completed above, per model, with its own
# checkpointing. resolved_mistral here is the combined result.
print(f'{len(resolved_mistral):,} resolved mistral pairs across {resolved_mistral["model"].nunique()} models')

In [ ]:
# prometheus judging already completed above, per model, with its own
# checkpointing. resolved_prometheus here is the combined result.
print(f'{len(resolved_prometheus):,} resolved prometheus pairs across {resolved_prometheus["model"].nunique()} models')

In [ ]:
print("--- mistral win rates ---")
winrates_mistral = judge.summarise_pairwise(resolved_mistral)
display(winrates_mistral)

print("\n--- prometheus win rates ---")
winrates_prometheus = judge.summarise_pairwise(resolved_prometheus)
display(winrates_prometheus)

print("\n--- do the two judges agree? ---")
agreement = judge.judge_agreement(resolved_mistral, resolved_prometheus)

In [ ]:
# Point estimates alone cannot support the claim. A decisive win rate of
# 0.667 built on 9 decided pairs is not distinguishable from one of 0.200
# built on 5. These tests resample each condition and report an interval on
# the difference against its own random control.
print("--- does each method actually beat its random control? ---")
beat_mistral = judge.compare_to_random(resolved_mistral)
display(beat_mistral[beat_mistral['beats_random']])

print("
--- conditions that also beat the unsteered baseline ---")
print("(lower bound of the interval above 0.5, not just above random)")
display(winrates_mistral[winrates_mistral['beats_baseline']])

print("
Read n_decided, not n_pairs. High tie rates mean a condition with")
print("200 pairs can still rest on very few actual decisions.")

## Step 11b: a third judge, from a different family

Mistral-Small and Prometheus 2 both satisfy the rule that no judge comes from a family under test. They do not satisfy independence: Prometheus 2 is fine-tuned from Mistral-7B, so both sit on Mistral pretraining. Two judges from one base family plausibly share cultural blind spots, which is the exact failure this project is about, so their agreeing with each other is weaker evidence than it looks.

Llama 3.1 8B is a genuinely different lineage. The mirror used here is ungated, so it needs no licence acceptance or login, though the weights are still under Meta's Llama 3.1 Community Licence rather than an OSI-approved one, which makes them open-weights rather than open-source.

This runs on a subset. A robustness check does not need every pair, and a third full pass would cost about as much as the first two together. The subset is taken at pair level, never at row level: a pair split across the boundary loses one of its two orders, and `resolve_pairwise` treats a half pair as incomplete and quietly resolves it to a tie.

On kappa. Cohen's kappa is defined for exactly two raters, so a third judge does not remove it, it just needs the generalisation. Fleiss' kappa extends the same chance-correction to any fixed number of raters, and both are reported below: pairwise Cohen between each pair of judges, and one Fleiss figure across all three.

In [ ]:
# Subset at pair level so both presentation orders of a pair stay together.
judge_subset = judge.sample_pairs(pairs_judged, n_pairs=300)

judged_llama = judge.run_judge(judge_subset, "llama", tag='third_judge')
resolved_llama = judge.resolve_pairwise(judged_llama, tag='third_judge')
judge.unload_judge()

print("\n--- llama win rates ---")
display(judge.summarise_pairwise(resolved_llama))

resolved_all = {
    'mistral': resolved_mistral,
    'prometheus': resolved_prometheus,
    'llama': resolved_llama,
}

# Cohen's kappa pairwise, then Fleiss across all three at once.
agreement_matrix = judge.judge_agreement_matrix(resolved_all)
fleiss = judge.fleiss_kappa(resolved_all)

print('\nmistral vs prometheus share a base family, so their agreement is the one')
print('to discount. Either of them against llama is the real test of whether the')
print('result survives changing who is asked.')

## Step 12: human review (later)

Exports a blinded sample so a person can score it the same way the LLM judges did, for judge-vs-human agreement. Not meant to be filled in right now, this cell just produces the file to come back to.

Open `human_review_main.csv`, fill in the `winner` column with A, B, or tie for as many rows as you have time for, save it, then run the merge cell below.

In [ ]:
human_review = judge.export_for_human_review(pairs_judged, tag=run.tag)
human_review.head()

In [ ]:
# run this once human_review_main.csv has been filled in
import pandas as pd

human_csv_path = cfg.EVAL_DIR / f"human_review_{run.tag}.csv"
if human_csv_path.exists() and pd.read_csv(human_csv_path)["winner"].astype(str).str.strip().ne("").any():
    print("--- human vs mistral ---")
    judge.merge_human_review(human_csv_path, resolved_mistral, tag=run.tag)
    print("\n--- human vs prometheus ---")
    judge.merge_human_review(human_csv_path, resolved_prometheus, tag=run.tag)
else:
    print("human_review_main.csv has no filled-in verdicts yet, nothing to compare")

## Step 13: download the results

Everything is already saved to Drive as it runs, so this step is only for
pulling a copy onto the machine you are sitting at.

Zips `results/` and downloads it through the browser. The PRISM dataset is
left out: it is 128 MB and downloads itself from Hugging Face on any fresh
run, so there is no reason to carry it around.

In [ ]:
import os
import zipfile

archive_path = '/content/miorpa_results.zip'

with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder, _, filenames in os.walk('results'):
        for filename in filenames:
            file_path = os.path.join(folder, filename)
            archive.write(file_path, file_path)

size_mb = os.path.getsize(archive_path) / 1e6
n_files = len(zipfile.ZipFile(archive_path).namelist())
print(f'{n_files:,} files, {size_mb:.1f} MB -> {archive_path}')

try:
    from google.colab import files

    files.download(archive_path)
except ImportError:
    print('not on Colab; the zip is at the path above')